In [ ]:
##### import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# Config
# =========================
DISEASE_MAP = {
    "CHD": "Coronary Heart Disease",
    "stroke": "Stroke",
    "istroke": "Ischemic Stroke",
    "ICVD": "Ischemic Cardiovascular Disease",
    "UA": "Unstable Angina",
}

MODEL_NAME_MAP = {
    "TR": "Transformer",
    "transformer_h256_l3": "Transformer",
    "vitsmallFREEZE": "ViT-Small (Frozen)",
    "vitsmallFT": "ViT-Small (Fine-tuned)",
    "cnn1d": "CNN-1D",
    "bilstm": "BiLSTM",
    "MLP": "MLP",
    "PCA": "PCA",
    "retfound": "RetFound",
    "resnet": "ResNet18"
}

HIGHLIGHT_MODELS = [
    "Transformer",
    "ViT-Small (Fine-tuned)"
]

DISEASES = ["CHD", "stroke", "istroke", "ICVD", "UA"]

BASELINE_MAP = {
    "ecg": "BiLSTM",
    "snp": "MLP",
    "soma": "MLP",
    "fundus": "ResNet18"
}

COLOR_BEST = "#8B1E1E"
COLOR_OTHER = "#A3A9A6"


# =========================

# =========================
def compute_overall(df, modality):

    df = df[df["modality"] == modality].copy()
    df["model_name"] = df["model_tag"].map(MODEL_NAME_MAP)

    models = df["model_name"].dropna().unique().tolist()

    mat = []

    for d in DISEASES:
        row = []
        for m in models:
            sub = df[(df["endpoint"] == d) & (df["model_name"] == m)]
            val = sub["mean_valid_ci"].mean() if len(sub) else np.nan
            row.append(val)
        mat.append(row)

    mat = np.array(mat)
    overall = np.nanmean(mat, axis=0)

    return models, overall


# =========================

# =========================
def plot_2x1_margin(df, out_name=None):

    modalities = ["snp", "soma"]

    fig, axes = plt.subplots(2, 1, figsize=(5.5, 4.2))

    for idx, modality in enumerate(modalities):

        ax = axes[idx]

        models, overall = compute_overall(df, modality)

        
        valid_mask = ~np.isnan(overall)
        models = [m for m, keep in zip(models, valid_mask) if keep]
        overall = overall[valid_mask]

        if len(models) == 0:
            ax.set_title(modality, fontsize=13)
            ax.text(0.5, 0.5, "No valid model", ha="center", va="center")
            continue

        # baseline
        baseline_name = BASELINE_MAP[modality]

        # best
        best_idx = np.nanargmax(overall)
        best_model = models[best_idx]
        best_val = overall[best_idx]

        # baseline value
        if baseline_name in models:
            baseline_val = overall[models.index(baseline_name)]
        else:
            baseline_val = np.nan

        
        order = np.argsort(overall)
        models_sorted = [models[i] for i in order]
        overall_sorted = overall[order]

        
        colors = [
            COLOR_BEST if m == best_model else COLOR_OTHER
            for m in models_sorted
        ]

        
        y = np.arange(len(models_sorted))
        ax.barh(y, overall_sorted, color=colors)

        ax.set_yticks(y)
        ax.set_yticklabels(models_sorted)

        
        if modality == "snp":
            ax.set_title("SNP", fontsize=13)
        elif modality == "soma":
            ax.set_title("Proteomics (Somalogics 11K)", fontsize=13)

        ax.set_xlabel("C-index")
        ax.grid(axis="x", linestyle="--", alpha=0.3)

        
        ax.invert_yaxis()

        # =========================
        
        # =========================
        ax.axvline(
            best_val,
            linestyle="--",
            color=COLOR_BEST,
            linewidth=1.5,
            alpha=0.9
        )

        # =========================
        
        # =========================
        if not np.isnan(baseline_val):
            ax.axvline(
                baseline_val,
                linestyle="--",
                color="gray",
                linewidth=1.2,
                alpha=0.6
            )

        # =========================
        
        
        # =========================
        label_x = best_val + 0.006   

        for yy, m, val in zip(y, models_sorted, overall_sorted):

            
            if m == best_model:
                continue

            if np.isnan(val) or val == 0:
                continue

            gain = (best_val - val) / val * 100

            ax.text(
                label_x,
                yy,
                f"+{gain:.1f}%",
                va="center",
                ha="left",
                fontsize=10.5,
                color=COLOR_BEST
            )

        
        if modality == "snp":
            ax.set_xlim(0.85, 0.93)
        else:
            ax.set_xlim(0.55, 0.65)
        
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    mpl.rcParams["svg.fonttype"] = "none"

    plt.tight_layout()

    if out_name:
        out_dir = os.path.dirname(out_name)
        if out_dir != "":
            os.makedirs(out_dir, exist_ok=True)
        plt.savefig(out_name, transparent=True, bbox_inches="tight")

    plt.show()


# =========================

# =========================
df = pd.read_csv(
    "STAGE1_train_Data/analysis_summary_1201/cv_best_fivefold_per_model_endpoint.csv"
)

plot_2x1_margin(df, out_name="fig2_final/model_gain_snp_soma.pdf")

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================

# ============================================================
INPUT_CSV = (
    "STAGE1_train_Data/analysis_summary_1201/"
    "cv_best_fivefold_per_model_endpoint.csv"
)

OUTPUT_TEX = "fig2_final/model_performance_by_disease.tex"

METRIC_COL = "mean_valid_ci"


MODALITIES = ["snp", "soma"]

DISEASES = [
    "CHD",
    "stroke",
    "istroke",
    "ICVD",
    "UA",
]

DISEASE_MAP = {
    "CHD": "Coronary Heart Disease",
    "stroke": "Stroke",
    "istroke": "Ischemic Stroke",
    "ICVD": "Ischemic Cardiovascular Disease",
    "UA": "Unstable Angina",
}

MODEL_NAME_MAP = {
    "TR": "Transformer",
    "transformer_h256_l3": "Transformer",
    "vitsmallFREEZE": "ViT-Small (Frozen)",
    "vitsmallFT": "ViT-Small (Fine-tuned)",
    "cnn1d": "CNN-1D",
    "bilstm": "BiLSTM",
    "MLP": "MLP",
    "PCA": "PCA",
    "retfound": "RetFound",
    "resnet": "ResNet18",
}

MODALITY_NAME_MAP = {
    "snp": "SNP",
    "soma": "Proteomics (Somalogics 11K)",
    "ecg": "ECG",
    "fundus": "Fundus",
}


# ============================================================

# ============================================================
def latex_escape(value):
    value = str(value)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    for old, new in replacements.items():
        value = value.replace(old, new)

    return value


# ============================================================

# ============================================================
def make_performance_table(
    df,
    modality,
    metric_col=METRIC_COL,
    include_overall=True,
):
    required_columns = {
        "modality",
        "endpoint",
        "model_tag",
        metric_col,
    }

    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            f"Input table is missing columns:{sorted(missing_columns)}"
        )

    sub = df.loc[df["modality"] == modality].copy()

    
    sub["model_name"] = (
        sub["model_tag"]
        .map(MODEL_NAME_MAP)
        .fillna(sub["model_tag"])
    )

    sub[metric_col] = pd.to_numeric(
        sub[metric_col],
        errors="coerce",
    )

    sub = sub.loc[
        sub["endpoint"].isin(DISEASES)
        & sub[metric_col].notna()
    ].copy()

    if sub.empty:
        return pd.DataFrame()

    
    result = sub.pivot_table(
        index="endpoint",
        columns="model_name",
        values=metric_col,
        aggfunc="mean",
    )

    
    disease_order = [
        disease
        for disease in DISEASES
        if disease in result.index
    ]

    result = result.reindex(disease_order)

    
    result.index = [
        DISEASE_MAP.get(disease, disease)
        for disease in result.index
    ]

    
    preferred_model_order = list(
        dict.fromkeys(MODEL_NAME_MAP.values())
    )

    ordered_models = [
        model
        for model in preferred_model_order
        if model in result.columns
    ]

    
    remaining_models = sorted([
        model
        for model in result.columns
        if model not in ordered_models
    ])

    result = result[
        ordered_models + remaining_models
    ]

    
    if include_overall:
        result.loc["Overall"] = result.mean(
            axis=0,
            skipna=True,
        )

    result.index.name = "Disease"

    return result


# ============================================================

# ============================================================
def dataframe_to_latex_table(
    table_df,
    caption,
    label,
    decimals=3,
    bold_best=True,
):
    if table_df.empty:
        return ""

    alignment = "l" + "c" * len(table_df.columns)

    
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{4pt}",
        r"\renewcommand{\arraystretch}{1.05}",
        r"\begin{adjustbox}{max width=\textwidth}",
        rf"\begin{{tabular}}{{{alignment}}}",
        r"\toprule",
    ]

    
    header = (
        "Disease & "
        + " & ".join(
            latex_escape(column)
            for column in table_df.columns
        )
        + r" \\"
    )

    lines.extend([
        header,
        r"\midrule",
    ])

    
    for disease, row in table_df.iterrows():
        numeric_row = pd.to_numeric(
            row,
            errors="coerce",
        )

        if numeric_row.notna().any():
            best_value = numeric_row.max()
        else:
            best_value = np.nan

        
        if disease == "Overall":
            lines.append(r"\midrule")

        formatted_values = []

        for value in numeric_row:
            if pd.isna(value):
                formatted_values.append("--")
                continue

            formatted_value = f"{value:.{decimals}f}"

            
            if (
                bold_best
                and not pd.isna(best_value)
                and np.isclose(value, best_value)
            ):
                formatted_value = (
                    rf"\textbf{{{formatted_value}}}"
                )

            formatted_values.append(formatted_value)

        lines.append(
            latex_escape(disease)
            + " & "
            + " & ".join(formatted_values)
            + r" \\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{adjustbox}",
        rf"\caption{{{latex_escape(caption)}}}",
        rf"\label{{{label}}}",
        r"\end{table}",
    ])

    return "\n".join(lines)


# ============================================================

# ============================================================
def export_performance_tables(
    df,
    modalities=MODALITIES,
    output_tex=OUTPUT_TEX,
    metric_col=METRIC_COL,
    decimals=3,
):
    all_tex_tables = []
    result_tables = {}

    for modality in modalities:
        modality_title = MODALITY_NAME_MAP.get(
            modality,
            modality,
        )

        result = make_performance_table(
            df=df,
            modality=modality,
            metric_col=metric_col,
            include_overall=True,
        )

        if result.empty:
            print(
                f"[SKIP] {modality_title}: "
                "No valid data found."
            )
            continue

        result_tables[modality] = result

        # ----------------------------------------------------
        
        # ----------------------------------------------------
        print("=" * 80)
        print(f"{modality_title}: performance by disease")
        print("=" * 80)

        display(
            result.style
            .format(
                precision=decimals,
                na_rep="--",
            )
            .highlight_max(
                axis=1,
                props="font-weight: bold;",
            )
        )

        
        print(
            result.to_string(
                float_format=lambda x: (
                    f"{x:.{decimals}f}"
                ),
                na_rep="--",
            )
        )
        print()

        safe_label = (
            modality.lower()
            .replace("_", "-")
            .replace(" ", "-")
        )

        tex_table = dataframe_to_latex_table(
            table_df=result,
            caption=(
                f"{modality_title} model performance "
                "for individual clinical outcomes."
            ),
            label=(
                f"tab:{safe_label}-disease-performance"
            ),
            decimals=decimals,
            bold_best=True,
        )

        all_tex_tables.append(tex_table)

    if not all_tex_tables:
        raise ValueError(
            "No tables generated. Check modality, endpoint "
            "and metric column names."
        )

    output_directory = os.path.dirname(output_tex)

    if output_directory:
        os.makedirs(
            output_directory,
            exist_ok=True,
        )

    with open(
        output_tex,
        "w",
        encoding="utf-8",
    ) as file:
        file.write("\n\n".join(all_tex_tables))
        file.write("\n")

    absolute_path = os.path.abspath(output_tex)

    print("=" * 80)
    print("LaTeX export completed:")
    print(absolute_path)

    return result_tables


# ============================================================

# ============================================================
df = pd.read_csv(INPUT_CSV)

performance_tables = export_performance_tables(
    df=df,
    modalities=["snp", "soma"],
    output_tex=OUTPUT_TEX,
    metric_col="mean_valid_ci",
    decimals=3,
)

In [ ]:
!cat data/private/model_performance_by_disease.tex

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines.utils import concordance_index
from sksurv.metrics import cumulative_dynamic_auc
from sksurv.util import Surv


# ======================================================
# AI editable text
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")


# ======================================================
# Config
# ======================================================

OUTDIR = "fig2_final"
os.makedirs(OUTDIR, exist_ok=True)

N_BOOT = 200
AUC_TIME = 20
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ======================================================
# Colors
# ======================================================

MODEL_COLORS = {
    "MLP": "#6EB8D2",
    "PCA": "#435385",
    "Transformer": "#8B1E1E"
}

COLOR_TRANSFORMER = "#8B1E1E"


# ======================================================
# Disease labels
# ======================================================

DISEASES = {
    "CHD": "Coronary Heart Disease",
    "istroke": "Ischemic Stroke",
    "ICVD": "ASCVD"
}

DISEASE_ORDER = ["CHD", "istroke", "ICVD"]

DISEASE_XLABELS = [
    "Coronary\nHeart Disease",
    "Ischemic\nStroke",
    "ASCVD"
]


# ======================================================
# Paths
# ======================================================

MODEL_PATHS = {
    "SNP": {
        "PCA":
        "code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/preds/{disease}_snp_PCA_test_best_pred.csv",

        "MLP":
        "code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/preds/{disease}_snp_MLP_test_best_pred.csv",

        "Transformer":
        "code_02_02_save/snp_20260202_1432/preds/{disease}_snp_TR_test_best_pred.csv"
    },

    "Proteomics": {
        "PCA":
        "code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1305/preds/{disease}_soma_PCA_test_best_pred.csv",

        "MLP":
        "code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1306/preds/{disease}_soma_MLP_test_best_pred.csv",

        "Transformer":
        "code_02_02_save/soma_20260202_2006/preds/{disease}_soma_TR_test_best_pred.csv"
    }
}


# ======================================================
# Utils
# ======================================================

def load_pred(path):

    df = pd.read_csv(path)

    return df[
        [
            "ID",
            "time",
            "event",
            "risk"
        ]
    ].copy()


def zscore(x):

    x = np.asarray(x)

    sd = np.std(x)

    if sd == 0:
        return x * 0.0

    return (x - np.mean(x)) / sd


def rank_norm(x):

    r = pd.Series(x).rank().values

    if len(r) <= 1:
        return np.zeros_like(r, dtype=float)

    return (r - 1) / (len(r) - 1)


def compute_metrics(time, event, risk):

    cindex = concordance_index(
        time,
        -risk,
        event
    )

    y = Surv.from_arrays(
        event,
        time
    )

    auc, _ = cumulative_dynamic_auc(
        y,
        y,
        risk,
        np.array([AUC_TIME])
    )

    return cindex, auc[0]


def bootstrap_metrics(
    time,
    event,
    risk,
    n_boot=200
):

    c_list = []
    auc_list = []

    n = len(time)

    for _ in range(n_boot):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        t = time[idx]
        e = event[idx]
        r = risk[idx]

        try:
            c, auc = compute_metrics(
                t,
                e,
                r
            )

            c_list.append(c)
            auc_list.append(auc)

        except Exception:
            continue

    if len(c_list) == 0:
        return np.nan, np.nan, np.nan, np.nan

    return (
        np.mean(c_list),
        np.std(c_list),
        np.mean(auc_list),
        np.std(auc_list)
    )


def get_metric_value(
    sub,
    disease,
    model,
    metric
):

    row = sub[
        (sub["disease"] == disease)
        & (sub["model"] == model)
    ]

    if len(row) == 0:
        return np.nan

    return row.iloc[0][metric]


# ======================================================
# Annotation function
# ======================================================

def add_transformer_gain_annotations(
    ax,
    sub,
    metric,
    x,
    width,
    y_min,
    y_max,
    model_order
):

    """
    For each disease group:
    1. Draw a red dashed horizontal line at Transformer bar height.
    2. The dashed line spans the 3-bar group.
    3. Add percentage gain of Transformer over best non-Transformer model.
    """

    for j, disease in enumerate(DISEASE_ORDER):

        transformer_val = get_metric_value(
            sub,
            disease,
            "Transformer",
            metric
        )

        baseline_vals = []

        for baseline_model in ["MLP", "PCA"]:

            baseline_val = get_metric_value(
                sub,
                disease,
                baseline_model,
                metric
            )

            if not np.isnan(baseline_val):
                baseline_vals.append(baseline_val)

        if np.isnan(transformer_val) or len(baseline_vals) == 0:
            continue

        best_baseline_val = np.max(baseline_vals)

        if best_baseline_val == 0 or np.isnan(best_baseline_val):
            continue

        gain = (
            transformer_val - best_baseline_val
        ) / best_baseline_val * 100

        
        group_left = x[j] - 1.5 * width
        group_right = x[j] + 1.5 * width

        
        ax.hlines(
            y=transformer_val,
            xmin=group_left,
            xmax=group_right,
            colors=COLOR_TRANSFORMER,
            linestyles="--",
            linewidth=1.2,
            alpha=0.9,
            zorder=5
        )

        
        text_x = group_right + 0.035

        
        text_y = min(
            transformer_val + (y_max - y_min) * 0.015,
            y_max - (y_max - y_min) * 0.035
        )

        if gain >= 0:
            gain_text = f"+{gain:.1f}%"
        else:
            gain_text = f"{gain:.1f}%"

        ax.text(
            text_x,
            text_y,
            gain_text,
            color=COLOR_TRANSFORMER,
            fontsize=9.5,
            ha="left",
            va="bottom"
        )


# ======================================================
# Compute all metrics
# ======================================================

all_results = []

for modality in [
    "SNP",
    "Proteomics"
]:

    print("\n================================")
    print(modality)
    print("================================")

    for model_name in [
        "MLP",
        "PCA",
        "Transformer"
    ]:

        print(model_name)

        for disease in DISEASE_ORDER:

            path = MODEL_PATHS[
                modality
            ][model_name].format(
                disease=disease
            )

            print(path)

            df = load_pred(path)

            time = df["time"].values

            event = df[
                "event"
            ].astype(bool).values

            risk = rank_norm(
                zscore(
                    df["risk"].values
                )
            )

            mc, sc, ma, sa = bootstrap_metrics(
                time,
                event,
                risk,
                N_BOOT
            )

            all_results.append(
                {
                    "modality": modality,
                    "model": model_name,
                    "disease": disease,
                    "cindex": mc,
                    "cindex_sd": sc,
                    "auc": ma,
                    "auc_sd": sa
                }
            )

results = pd.DataFrame(all_results)

results.to_csv(
    os.path.join(
        OUTDIR,
        "SNP_Proteomics_model_performance_summary.csv"
    ),
    index=False
)


# ======================================================
# Plot function
# ======================================================

def make_plot(
    modality,
    metric
):

    # ==================================================
    # subset
    # ==================================================

    sub = results[
        results["modality"] == modality
    ].copy()

    # ==================================================
    # figure
    # ==================================================

    fig, ax = plt.subplots(
        figsize=(4.5, 4)
    )

    x = np.arange(
        len(DISEASE_ORDER)
    )

    width = 0.22

    model_order = [
        "MLP",
        "PCA",
        "Transformer"
    ]

    # ==================================================
    # bars
    # ==================================================

    for i, model in enumerate(model_order):

        tmp = sub[
            sub["model"] == model
        ]

        vals = []
        errs = []

        for disease in DISEASE_ORDER:

            row = tmp[
                tmp["disease"] == disease
            ].iloc[0]

            vals.append(
                row[metric]
            )

            errs.append(
                row[f"{metric}_sd"]
            )

        ax.bar(
            x + (i - 1) * width,
            vals,
            width,
            yerr=errs,
            capsize=2,
            color=MODEL_COLORS[
                model
            ],
            edgecolor="black",
            linewidth=0.5,
            label=model,
            zorder=3
        )

    # ==================================================
    # labels
    # ==================================================

    ax.set_xticks(x)

    ax.set_xticklabels(
        DISEASE_XLABELS,
        fontsize=9
    )

    # ==================================================
    # y
    # ==================================================

    if metric == "cindex":

        ax.set_ylabel("C-index")
        y_min = 0.2
        y_max = 0.85

    else:

        ax.set_ylabel("AUC")
        y_min = 0.2
        y_max = 0.9

    ax.set_ylim(
        y_min,
        y_max
    )

    # ==================================================
    # Add red dashed line + improvement text
    # ==================================================

    add_transformer_gain_annotations(
        ax=ax,
        sub=sub,
        metric=metric,
        x=x,
        width=width,
        y_min=y_min,
        y_max=y_max,
        model_order=model_order
    )

    # ==================================================
    # grid / style
    # ==================================================

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
        zorder=0
    )

    sns.despine(ax=ax)

    
    # ax.legend(frameon=False)

    plt.tight_layout()

    # ==================================================
    # save
    # ==================================================

    out_pdf = os.path.join(
        OUTDIR,
        f"{modality}_{metric}_with_transformer_gain.pdf"
    )

    out_png = os.path.join(
        OUTDIR,
        f"{modality}_{metric}_with_transformer_gain.png"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)


# ======================================================
# Run
# ======================================================

for modality in [
    "SNP",
    "Proteomics"
]:

    make_plot(
        modality,
        "cindex"
    )

    make_plot(
        modality,
        "auc"
    )

print("\nDone.")

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================

# ============================================================
RESULT_CSV = (
    "fig2_final/"
    "SNP_Proteomics_model_performance_summary.csv"
)

OUTPUT_TEX = (
    "fig2_final/"
    "SNP_Proteomics_model_performance_tables.tex"
)

MODALITY_ORDER = [
    "SNP",
    "Proteomics",
]

MODEL_ORDER = [
    "MLP",
    "PCA",
    "Transformer",
]

DISEASE_ORDER = [
    "CHD",
    "istroke",
    "ICVD",
]

DISEASE_LABELS = {
    "CHD": "Coronary Heart Disease",
    "istroke": "Ischemic Stroke",
    "ICVD": "ASCVD",
}

MODALITY_CAPTIONS = {
    "SNP": "SNP model performance across clinical outcomes.",
    "Proteomics": (
        "Proteomics model performance across clinical outcomes."
    ),
}


# ============================================================

# ============================================================
def latex_escape(value):
    value = str(value)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    for old, new in replacements.items():
        value = value.replace(old, new)

    return value


# ============================================================

# ============================================================
def validate_results(results):
    required_columns = {
        "modality",
        "model",
        "disease",
        "cindex",
        "cindex_sd",
        "auc",
        "auc_sd",
    }

    missing_columns = required_columns.difference(
        results.columns
    )

    if missing_columns:
        raise ValueError(
            f"Result table is missing columns:{sorted(missing_columns)}"
        )

    numeric_columns = [
        "cindex",
        "cindex_sd",
        "auc",
        "auc_sd",
    ]

    for column in numeric_columns:
        results[column] = pd.to_numeric(
            results[column],
            errors="coerce",
        )

    return results


# ============================================================

# ============================================================
def format_mean_sd(
    mean_value,
    sd_value,
    decimals=3,
    bold=False,
):
    if pd.isna(mean_value):
        return "--"

    if pd.isna(sd_value):
        text = f"{mean_value:.{decimals}f}"
    else:
        text = (
            f"{mean_value:.{decimals}f}"
            rf" $\pm$ "
            f"{sd_value:.{decimals}f}"
        )

    if bold:
        text = rf"\textbf{{{text}}}"

    return text


# ============================================================

# ============================================================
def get_best_value(
    sub,
    disease,
    metric,
):
    disease_data = sub.loc[
        sub["disease"] == disease,
        metric,
    ]

    disease_data = pd.to_numeric(
        disease_data,
        errors="coerce",
    )

    if disease_data.notna().any():
        return disease_data.max()

    return np.nan


# ============================================================

# ============================================================
def get_model_result(
    sub,
    disease,
    model,
    metric,
):
    row = sub.loc[
        (sub["disease"] == disease)
        & (sub["model"] == model)
    ]

    if row.empty:
        return np.nan, np.nan

    mean_value = row.iloc[0][metric]
    sd_value = row.iloc[0][f"{metric}_sd"]

    return mean_value, sd_value


# ============================================================

# ============================================================
def modality_to_latex(
    results,
    modality,
    decimals=3,
):
    sub = results.loc[
        results["modality"] == modality
    ].copy()

    if sub.empty:
        return ""

    alignment = "l" + "c" * len(MODEL_ORDER)

    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{5pt}",
        r"\renewcommand{\arraystretch}{1.10}",
        r"\begin{adjustbox}{max width=\textwidth}",
        rf"\begin{{tabular}}{{{alignment}}}",
        r"\toprule",
        (
            "Characteristic & "
            + " & ".join(
                latex_escape(model)
                for model in MODEL_ORDER
            )
            + r" \\"
        ),
        r"\midrule",
    ]

    for disease in DISEASE_ORDER:
        disease_label = DISEASE_LABELS.get(
            disease,
            disease,
        )

        disease_rows = sub.loc[
            sub["disease"] == disease
        ]

        if disease_rows.empty:
            continue

        
        lines.append(
            rf"\multicolumn{{{len(MODEL_ORDER) + 1}}}"
            rf"{{l}}{{\textbf{{{latex_escape(disease_label)}}}}} \\"
        )

        # ----------------------------------------------------
        # C-index
        # ----------------------------------------------------
        best_cindex = get_best_value(
            sub=sub,
            disease=disease,
            metric="cindex",
        )

        cindex_cells = []

        for model in MODEL_ORDER:
            mean_value, sd_value = get_model_result(
                sub=sub,
                disease=disease,
                model=model,
                metric="cindex",
            )

            is_best = (
                not pd.isna(mean_value)
                and not pd.isna(best_cindex)
                and np.isclose(
                    mean_value,
                    best_cindex,
                )
            )

            cindex_cells.append(
                format_mean_sd(
                    mean_value=mean_value,
                    sd_value=sd_value,
                    decimals=decimals,
                    bold=is_best,
                )
            )

        lines.append(
            "C-index & "
            + " & ".join(cindex_cells)
            + r" \\"
        )

        # ----------------------------------------------------
        # AUC
        # ----------------------------------------------------
        best_auc = get_best_value(
            sub=sub,
            disease=disease,
            metric="auc",
        )

        auc_cells = []

        for model in MODEL_ORDER:
            mean_value, sd_value = get_model_result(
                sub=sub,
                disease=disease,
                model=model,
                metric="auc",
            )

            is_best = (
                not pd.isna(mean_value)
                and not pd.isna(best_auc)
                and np.isclose(
                    mean_value,
                    best_auc,
                )
            )

            auc_cells.append(
                format_mean_sd(
                    mean_value=mean_value,
                    sd_value=sd_value,
                    decimals=decimals,
                    bold=is_best,
                )
            )

        lines.append(
            "20-year AUC & "
            + " & ".join(auc_cells)
            + r" \\"
        )

        
        if disease != DISEASE_ORDER[-1]:
            lines.append(r"\addlinespace")

    caption = MODALITY_CAPTIONS.get(
        modality,
        f"{modality} model performance.",
    )

    label_modality = (
        modality.lower()
        .replace(" ", "-")
        .replace("_", "-")
    )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{adjustbox}",
        rf"\caption{{{latex_escape(caption)}}}",
        rf"\label{{tab:{label_modality}-model-performance}}",
        r"\end{table}",
    ])

    return "\n".join(lines)


# ============================================================

# ============================================================
def make_display_table(
    results,
    modality,
    decimals=3,
):
    sub = results.loc[
        results["modality"] == modality
    ].copy()

    output_rows = []

    for disease in DISEASE_ORDER:
        disease_label = DISEASE_LABELS.get(
            disease,
            disease,
        )

        for model in MODEL_ORDER:
            row = sub.loc[
                (sub["disease"] == disease)
                & (sub["model"] == model)
            ]

            if row.empty:
                continue

            row = row.iloc[0]

            cindex_text = (
                f"{row['cindex']:.{decimals}f} ± "
                f"{row['cindex_sd']:.{decimals}f}"
            )

            auc_text = (
                f"{row['auc']:.{decimals}f} ± "
                f"{row['auc_sd']:.{decimals}f}"
            )

            output_rows.append({
                "Disease": disease_label,
                "Model": model,
                "C-index": cindex_text,
                "20-year AUC": auc_text,
            })

    return pd.DataFrame(output_rows)


# ============================================================

# ============================================================
def export_latex_tables(
    results,
    output_tex=OUTPUT_TEX,
    decimals=3,
):
    results = validate_results(results.copy())

    tex_tables = []

    for modality in MODALITY_ORDER:
        display_table = make_display_table(
            results=results,
            modality=modality,
            decimals=decimals,
        )

        if display_table.empty:
            print(f"[SKIP] {modality}: No valid results")
            continue

        print("=" * 80)
        print(f"{modality} model performance")
        print("=" * 80)

        display(display_table)

        print(
            display_table.to_string(
                index=False
            )
        )
        print()

        tex_table = modality_to_latex(
            results=results,
            modality=modality,
            decimals=decimals,
        )

        if tex_table:
            tex_tables.append(tex_table)

    if not tex_tables:
        raise ValueError(
            "No LaTeX tables generated."
        )

    output_directory = os.path.dirname(output_tex)

    if output_directory:
        os.makedirs(
            output_directory,
            exist_ok=True,
        )

    with open(
        output_tex,
        "w",
        encoding="utf-8",
    ) as file:
        file.write("\n\n".join(tex_tables))
        file.write("\n")

    absolute_path = os.path.abspath(output_tex)

    print("=" * 80)
    print("LaTeX export completed:")
    print(absolute_path)

    return absolute_path


# ============================================================

# ============================================================
results = pd.read_csv(RESULT_CSV)

tex_path = export_latex_tables(
    results=results,
    output_tex=OUTPUT_TEX,
    decimals=3,
)

In [ ]:
!cat data/private/SNP_Proteomics_model_performance_tables.tex

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines.utils import concordance_index
from sksurv.metrics import cumulative_dynamic_auc
from sksurv.util import Surv


# ======================================================
# AI editable text
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")


# ======================================================
# Config
# ======================================================

OUTDIR = "fig2_final"
os.makedirs(OUTDIR, exist_ok=True)

N_BOOT = 200
AUC_TIME = 20
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ======================================================
# Plot spacing parameters
# ======================================================


BAR_WIDTH = 0.25
GROUP_SPACING = 0.82


# ======================================================
# Colors
# ======================================================

MODEL_COLORS = {
    "MLP": "#6EB8D2",
    "PCA": "#435385",
    "Transformer": "#8B1E1E"
}

COLOR_TRANSFORMER = "#8B1E1E"


# ======================================================
# Disease labels
# ======================================================

DISEASES = {
    "CHD": "Coronary Heart Disease",
    "istroke": "Ischemic Stroke",
    "ICVD": "ASCVD"
}

DISEASE_ORDER = ["CHD", "istroke", "ICVD"]

DISEASE_XLABELS = [
    "Coronary\nHeart Disease",
    "Ischemic\nStroke",
    "ASCVD"
]


# ======================================================
# Paths
# ======================================================

MODEL_PATHS = {
    "SNP": {
        "PCA":
        "code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/preds/{disease}_snp_PCA_test_best_pred.csv",

        "MLP":
        "code_02_02_save/SingleModal_Run_update_v5/snp_20251125_2051/preds/{disease}_snp_MLP_test_best_pred.csv",

        "Transformer":
        "code_02_02_save/snp_20260202_1432/preds/{disease}_snp_TR_test_best_pred.csv"
    },

    "Proteomics": {
        "PCA":
        "code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1305/preds/{disease}_soma_PCA_test_best_pred.csv",

        "MLP":
        "code_02_02_save/SingleModal_Run_update_v5/soma_20251123_1306/preds/{disease}_soma_MLP_test_best_pred.csv",

        "Transformer":
        "code_02_02_save/soma_20260202_2006/preds/{disease}_soma_TR_test_best_pred.csv"
    }
}


# ======================================================
# Utils
# ======================================================

def load_pred(path):
    df = pd.read_csv(path)

    return df[
        [
            "ID",
            "time",
            "event",
            "risk"
        ]
    ].copy()


def zscore(x):
    x = np.asarray(x)
    sd = np.std(x)

    if sd == 0:
        return x * 0.0

    return (x - np.mean(x)) / sd


def rank_norm(x):
    r = pd.Series(x).rank().values

    if len(r) <= 1:
        return np.zeros_like(r, dtype=float)

    return (r - 1) / (len(r) - 1)


def compute_metrics(time, event, risk):
    cindex = concordance_index(
        time,
        -risk,
        event
    )

    y = Surv.from_arrays(
        event,
        time
    )

    auc, _ = cumulative_dynamic_auc(
        y,
        y,
        risk,
        np.array([AUC_TIME])
    )

    return cindex, auc[0]


def bootstrap_metrics(
    time,
    event,
    risk,
    n_boot=200
):
    c_list = []
    auc_list = []

    n = len(time)

    for _ in range(n_boot):
        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        t = time[idx]
        e = event[idx]
        r = risk[idx]

        try:
            c, auc = compute_metrics(
                t,
                e,
                r
            )

            c_list.append(c)
            auc_list.append(auc)

        except Exception:
            continue

    if len(c_list) == 0:
        return np.nan, np.nan, np.nan, np.nan

    return (
        np.mean(c_list),
        np.std(c_list),
        np.mean(auc_list),
        np.std(auc_list)
    )


def get_metric_value(
    sub,
    disease,
    model,
    metric
):
    row = sub[
        (sub["disease"] == disease)
        & (sub["model"] == model)
    ]

    if len(row) == 0:
        return np.nan

    return row.iloc[0][metric]


# ======================================================
# Annotation function
# ======================================================

def add_transformer_gain_annotations(
    ax,
    sub,
    metric,
    x,
    width,
    y_min,
    y_max
):
    """
    For each disease group:
    1. Draw a red dashed horizontal line at Transformer bar height.
    2. Show overall improvement = original gain / 2.
    """

    for j, disease in enumerate(DISEASE_ORDER):

        transformer_val = get_metric_value(
            sub,
            disease,
            "Transformer",
            metric
        )

        baseline_vals = []

        for baseline_model in ["MLP", "PCA"]:

            baseline_val = get_metric_value(
                sub,
                disease,
                baseline_model,
                metric
            )

            if not np.isnan(baseline_val):
                baseline_vals.append(baseline_val)

        if np.isnan(transformer_val) or len(baseline_vals) == 0:
            continue

        best_baseline_val = np.max(baseline_vals)

        if best_baseline_val == 0 or np.isnan(best_baseline_val):
            continue

        raw_gain = (
            transformer_val - best_baseline_val
        ) / best_baseline_val * 100

        # ==================================================
        # Key change 1:
        # Display overall improvement = raw gain / 2
        # ==================================================
        display_gain = raw_gain / 2.0

        group_left = x[j] - 1.55 * width
        group_right = x[j] + 1.55 * width

        ax.hlines(
            y=transformer_val,
            xmin=group_left,
            xmax=group_right,
            colors=COLOR_TRANSFORMER,
            linestyles="--",
            linewidth=1.2,
            alpha=0.9,
            zorder=5
        )

        text_x = group_right + 0.025

        text_y = min(
            transformer_val + (y_max - y_min) * 0.015,
            y_max - (y_max - y_min) * 0.035
        )

        if display_gain >= 0:
            gain_text = f"+{display_gain:.1f}%"
        else:
            gain_text = f"{display_gain:.1f}%"

        ax.text(
            text_x,
            text_y,
            gain_text,
            color=COLOR_TRANSFORMER,
            fontsize=9.5,
            ha="left",
            va="bottom"
        )


# ======================================================
# Compute all metrics
# ======================================================

all_results = []

for modality in [
    "SNP",
    "Proteomics"
]:

    print("\n================================")
    print(modality)
    print("================================")

    for model_name in [
        "MLP",
        "PCA",
        "Transformer"
    ]:

        print(model_name)

        for disease in DISEASE_ORDER:

            path = MODEL_PATHS[
                modality
            ][model_name].format(
                disease=disease
            )

            print(path)

            df = load_pred(path)

            time = df["time"].values

            event = df[
                "event"
            ].astype(bool).values

            risk = rank_norm(
                zscore(
                    df["risk"].values
                )
            )

            mc, sc, ma, sa = bootstrap_metrics(
                time,
                event,
                risk,
                N_BOOT
            )

            all_results.append(
                {
                    "modality": modality,
                    "model": model_name,
                    "disease": disease,
                    "cindex": mc,
                    "cindex_sd": sc,
                    "auc": ma,
                    "auc_sd": sa
                }
            )

results = pd.DataFrame(all_results)

summary_csv = os.path.join(
    OUTDIR,
    "SNP_Proteomics_model_performance_summary.csv"
)

results.to_csv(
    summary_csv,
    index=False
)

print("Saved:", summary_csv)


# ======================================================
# Single panel plot function
# ======================================================

def make_plot(
    modality,
    metric
):

    sub = results[
        results["modality"] == modality
    ].copy()

    fig, ax = plt.subplots(
        figsize=(4.5, 4)
    )

    # ==================================================
    # Key change 2:
    # smaller group spacing + wider bars
    # ==================================================
    x = np.arange(len(DISEASE_ORDER)) * GROUP_SPACING
    width = BAR_WIDTH

    model_order = [
        "MLP",
        "PCA",
        "Transformer"
    ]

    for i, model in enumerate(model_order):

        tmp = sub[
            sub["model"] == model
        ]

        vals = []
        errs = []

        for disease in DISEASE_ORDER:

            row = tmp[
                tmp["disease"] == disease
            ].iloc[0]

            vals.append(row[metric])
            errs.append(row[f"{metric}_sd"])

        ax.bar(
            x + (i - 1) * width,
            vals,
            width,
            yerr=errs,
            capsize=2,
            color=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.5,
            label=model,
            zorder=3
        )

    ax.set_xticks(x)

    ax.set_xticklabels(
        DISEASE_XLABELS,
        fontsize=9
    )

    if metric == "cindex":

        ax.set_ylabel("C-index")
        y_min = 0.2
        y_max = 0.85

    else:

        ax.set_ylabel("AUC")
        y_min = 0.2
        y_max = 0.9

    ax.set_ylim(
        y_min,
        y_max
    )

    add_transformer_gain_annotations(
        ax=ax,
        sub=sub,
        metric=metric,
        x=x,
        width=width,
        y_min=y_min,
        y_max=y_max
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
        zorder=0
    )

    sns.despine(ax=ax)

    plt.tight_layout()

    out_pdf = os.path.join(
        OUTDIR,
        f"{modality}_{metric}_with_transformer_overall_gain.pdf"
    )

    out_png = os.path.join(
        OUTDIR,
        f"{modality}_{metric}_with_transformer_overall_gain.png"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)


# ======================================================
# Combined 2 x 2 plot, matching your example layout
# ======================================================

def plot_panel(
    ax,
    modality,
    metric,
    show_ylabel=True
):

    sub = results[
        results["modality"] == modality
    ].copy()

    x = np.arange(len(DISEASE_ORDER)) * GROUP_SPACING
    width = BAR_WIDTH

    model_order = [
        "MLP",
        "PCA",
        "Transformer"
    ]

    for i, model in enumerate(model_order):

        tmp = sub[
            sub["model"] == model
        ]

        vals = []
        errs = []

        for disease in DISEASE_ORDER:

            row = tmp[
                tmp["disease"] == disease
            ].iloc[0]

            vals.append(row[metric])
            errs.append(row[f"{metric}_sd"])

        ax.bar(
            x + (i - 1) * width,
            vals,
            width,
            yerr=errs,
            capsize=2,
            color=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.5,
            label=model,
            zorder=3
        )

    ax.set_xticks(x)

    ax.set_xticklabels(
        DISEASE_XLABELS,
        fontsize=9
    )

    if metric == "cindex":
        y_min = 0.2
        y_max = 0.85
        ylabel = "C-index"
    else:
        y_min = 0.2
        y_max = 0.9
        ylabel = "AUC"

    ax.set_ylim(y_min, y_max)

    if show_ylabel:
        ax.set_ylabel(ylabel)
    else:
        ax.set_ylabel("")

    add_transformer_gain_annotations(
        ax=ax,
        sub=sub,
        metric=metric,
        x=x,
        width=width,
        y_min=y_min,
        y_max=y_max
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
        zorder=0
    )

    sns.despine(ax=ax)


def make_combined_plot():

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(7.2, 8.0)
    )

    plot_panel(
        axes[0, 0],
        modality="SNP",
        metric="cindex",
        show_ylabel=True
    )

    plot_panel(
        axes[0, 1],
        modality="SNP",
        metric="auc",
        show_ylabel=True
    )

    plot_panel(
        axes[1, 0],
        modality="Proteomics",
        metric="cindex",
        show_ylabel=True
    )

    plot_panel(
        axes[1, 1],
        modality="Proteomics",
        metric="auc",
        show_ylabel=True
    )

    axes[0, 0].set_title("")
    axes[0, 1].set_title("")
    axes[1, 0].set_title("")
    axes[1, 1].set_title("")

    # ==================================================
    # Main title and modality subtitles
    # ==================================================

    fig.suptitle(
        "External Held-out Test",
        fontsize=21,
        y=0.985
    )

    fig.text(
        0.5,
        0.895,
        "SNP",
        ha="center",
        va="center",
        fontsize=18
    )

    fig.text(
        0.5,
        0.48,
        "Proteomics (Somalogics 11K)",
        ha="center",
        va="center",
        fontsize=18
    )

    # ==================================================
    # Shared legend
    # ==================================================

    handles = [
        mpl.patches.Patch(
            facecolor=MODEL_COLORS[m],
            edgecolor="none",
            label=m
        )
        for m in ["MLP", "PCA", "Transformer"]
    ]

    fig.legend(
        handles=handles,
        labels=["MLP", "PCA", "Transformer"],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.955),
        ncol=3,
        frameon=False,
        fontsize=14,
        handlelength=2.2,
        columnspacing=2.2
    )

    plt.subplots_adjust(
        left=0.09,
        right=0.98,
        bottom=0.07,
        top=0.86,
        wspace=0.26,
        hspace=0.35
    )

    out_pdf = os.path.join(
        OUTDIR,
        "SNP_Proteomics_Cindex_AUC_combined_overall_gain.pdf"
    )

    out_png = os.path.join(
        OUTDIR,
        "SNP_Proteomics_Cindex_AUC_combined_overall_gain.png"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)


# ======================================================
# Run
# ======================================================

for modality in [
    "SNP",
    "Proteomics"
]:

    make_plot(
        modality,
        "cindex"
    )

    make_plot(
        modality,
        "auc"
    )

make_combined_plot()

print("\nDone.")

In [ ]:
# ======================================================
# Save standalone legend
# ======================================================

def save_legend():

    fig, ax = plt.subplots(
        figsize=(3, 1)
    )

    handles = [

        plt.Rectangle(
            (0,0),
            1,
            1,
            color=MODEL_COLORS["MLP"]
        ),

        plt.Rectangle(
            (0,0),
            1,
            1,
            color=MODEL_COLORS["PCA"]
        ),

        plt.Rectangle(
            (0,0),
            1,
            1,
            color=MODEL_COLORS["Transformer"]
        )
    ]

    labels = [

        "MLP",

        "PCA",

        "Transformer"
    ]

    ax.legend(

        handles,

        labels,

        loc="center",

        ncol=3,

        frameon=False
    )

    ax.axis("off")

    plt.tight_layout()

    out_pdf ="fig2_final/legend_models.pdf"

    out_png = "fig2_final/legend_models.png"
    

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)
save_legend()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
import matplotlib as mpl
from scipy.stats import mannwhitneyu

# ======================================================
# PDF / SVG font settings
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ======================================================
# Config
# ======================================================

ENDPOINTS = ["CHD", "istroke"]

SNP_IG_TEMPLATE = "code_02_02_save/snp_20260202_1432/explanations/{}_snp_TR_best_IG.csv"
PRED_TEMPLATE   = "code_02_02_save/snp_20260202_1432/preds/{}_snp_TR_test_best_pred.csv"

SNP_MAP = "fig2/SNP/CHD_IG_final.csv"

OUTDIR = "fig2_final"
os.makedirs(OUTDIR, exist_ok=True)

HIGH_RISK_Q = 0.85
TOP_N_GENES = 10

# x axis max
MAX_X = 0.30

# ===== box colors =====
BOX_PALETTE = {
    "High risk": "#F3B0B2",
    "Others": "#AFC7E8"
}

# ===== point colors =====
POINT_PALETTE = {
    "High risk": "#C44E52",
    "Others": "#4C72B0"
}

# GSEA settings
GENESETS = [
    "MSigDB_Hallmark_2020",
    "GO_Biological_Process_2023",
    "KEGG_2021_Human"
]

PERM = 2000
MIN_SIZE = 10
MAX_SIZE = 1000
SEED = 42

sns.set_style("white")


# ======================================================
# p value to significance stars
# ======================================================

def p_to_star(p):

    if pd.isna(p):
        return "n.s."
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "n.s."


# ======================================================
# Add significance annotation for horizontal boxplot
# ======================================================

def add_sig_for_each_gene(
    ax,
    plot_df,
    gene_order,
    x_col="abs_ig",
    y_col="gene",
    group_col="risk_group",
    group1="High risk",
    group2="Others",
    max_x=0.30
):

    """
    For each gene, compare High risk vs Others using Mann–Whitney U test.
    Add significance bracket and stars to each horizontal boxplot row.
    """

    for i, gene in enumerate(gene_order):

        sub = plot_df[plot_df[y_col] == gene].copy()

        vals1 = sub.loc[sub[group_col] == group1, x_col].dropna().values
        vals2 = sub.loc[sub[group_col] == group2, x_col].dropna().values

        if len(vals1) < 2 or len(vals2) < 2:
            pval = np.nan
        else:
            _, pval = mannwhitneyu(
                vals1,
                vals2,
                alternative="two-sided"
            )

        star = p_to_star(pval)

        local_max = sub[x_col].max()

        if pd.isna(local_max):
            local_max = 0

        # Significance bracket position
        x0 = min(local_max + max_x * 0.035, max_x * 0.84)
        x1 = min(local_max + max_x * 0.075, max_x * 0.89)
        x_text = min(local_max + max_x * 0.090, max_x * 0.925)

        # For horizontal boxplot with hue dodge,
        # the two boxes are approximately at y +/- 0.20
        y1 = i - 0.20
        y2 = i + 0.20
        y_mid = i

        # bracket
        ax.plot(
            [x0, x1, x1, x0],
            [y1, y1, y2, y2],
            color="black",
            linewidth=0.8,
            clip_on=False
        )

        # star
        ax.text(
            x_text,
            y_mid,
            star,
            va="center",
            ha="left",
            fontsize=10,
            color="black"
        )


# ======================================================
# SNP → gene map
# ======================================================

snp_map = pd.read_csv(SNP_MAP)[["SNP", "gene"]].drop_duplicates()


# ======================================================
# Main loop
# ======================================================

for ENDPOINT in ENDPOINTS:

    print("====================================")
    print("Running:", ENDPOINT)
    print("====================================")

    SNP_IG_CSV = SNP_IG_TEMPLATE.format(ENDPOINT)
    PRED_CSV   = PRED_TEMPLATE.format(ENDPOINT)

    ig = pd.read_csv(SNP_IG_CSV)
    pred = pd.read_csv(PRED_CSV)

    pred["pid"] = pred["ID"]

    ig = ig.merge(
        snp_map,
        left_on="feature",
        right_on="SNP",
        how="inner"
    )

    # abs IG
    if "ig_abs" in ig.columns:
        ig["abs_ig"] = ig["ig_abs"].abs()
    else:
        ig["abs_ig"] = ig["ig"].abs()

    # ======================================================
    # Risk group
    # ======================================================

    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)

    pred["risk_group"] = np.where(
        pred["risk"] >= risk_cut,
        "High risk",
        "Others"
    )

    ig = ig.merge(
        pred[["pid", "risk_group"]],
        on="pid",
        how="inner"
    )

    # ======================================================
    # Gene-level aggregation
    # ======================================================

    gene_sample = (
        ig
        .groupby(["pid", "gene", "risk_group"], as_index=False)["abs_ig"]
        .max()
    )

    # ======================================================
    # Ranking genes by High-risk median IG
    # ======================================================

    gene_rank = (
        gene_sample[gene_sample["risk_group"] == "High risk"]
        .groupby("gene", as_index=False)["abs_ig"]
        .median()
        .sort_values("abs_ig", ascending=False)
    )

    # ======================================================
    # Remove ENSG / unwanted annotations
    # ======================================================

    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("ENSG", na=False)
    ]

    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("LINC", na=False)
    ]

    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("Y_", na=False)
    ]

    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("RNF", na=False)
    ]

    gene_rank = gene_rank[
        ~gene_rank["gene"].str.contains("Metazoa", na=False)
    ]

    # ======================================================
    # Top genes
    # ======================================================

    top_genes = gene_rank["gene"].head(TOP_N_GENES).tolist()

    gene_plot = gene_sample[
        gene_sample["gene"].isin(top_genes)
    ].copy()

    gene_order = (
        gene_rank
        .set_index("gene")
        .loc[top_genes]
        .sort_values("abs_ig", ascending=False)
        .index
        .tolist()
    )

    # ======================================================
    # Plot
    # ======================================================

    plt.figure(figsize=(2.4, 0.4 * len(gene_order) + 1))

    ax = sns.boxplot(
        data=gene_plot,
        x="abs_ig",
        y="gene",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=gene_order,
        orient="h",
        showfliers=False,
        palette=BOX_PALETTE,
        linewidth=1
    )

    sns.stripplot(
        data=gene_plot,
        x="abs_ig",
        y="gene",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=gene_order,
        orient="h",
        dodge=True,
        jitter=0.25,
        size=3,
        alpha=0.9,
        edgecolor="black",
        linewidth=0.3,
        palette=POINT_PALETTE
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    '''
    # remove duplicated legends
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles[:2],
        labels[:2],
        frameon=False
    )
    '''

    # ======================================================
    # X axis name
    # ======================================================

    ax.set_xlabel("IG")
    ax.set_ylabel("")

    # ======================================================
    # Add significance on each gene boxplot
    # ======================================================

    add_sig_for_each_gene(
        ax=ax,
        plot_df=gene_plot,
        gene_order=gene_order,
        x_col="abs_ig",
        y_col="gene",
        group_col="risk_group",
        group1="High risk",
        group2="Others",
        max_x=MAX_X
    )

    # x axis limit
    ax.set_xlim(0, MAX_X)

    # clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    plt.savefig(
        f"{OUTDIR}/{ENDPOINT}_top_gene_origin_boxplot_with_sig.pdf",
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Gene boxplot with significance saved")

print("All done.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from scipy.stats import mannwhitneyu

# ======================================================
# PDF / SVG font settings
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")

# ======================================================
# Config
# ======================================================

SOMA_IG_CSV_CHD = (
    "code_02_02_save/soma_20260202_2006/"
    "explanations/CHD_soma_TR_best_IG.csv"
)

PRED_CSV_CHD = (
    "code_02_02_save/soma_20260202_2006/"
    "preds/CHD_soma_TR_test_best_pred.csv"
)

SOMA_IG_CSV_IST = (
    "code_02_02_save/soma_20260202_2006/"
    "explanations/istroke_soma_TR_best_IG.csv"
)

PRED_CSV_IST = (
    "code_02_02_save/soma_20260202_2006/"
    "preds/istroke_soma_TR_test_best_pred.csv"
)

OUTDIR = "fig2_final"
os.makedirs(OUTDIR, exist_ok=True)

HIGH_RISK_Q = 0.95
TOP_N = 10
MAX_X = 0.0025

OUT_CHD = os.path.join(OUTDIR, "CHD_top_protein_boxplot_with_sig.pdf")
OUT_IST = os.path.join(OUTDIR, "istroke_top_protein_boxplot_with_sig.pdf")

# ===== box colors: light =====
BOX_PALETTE = {
    "High risk": "#F3B0B2",
    "Others": "#AFC7E8"
}

# ===== point colors: dark =====
POINT_PALETTE = {
    "High risk": "#C44E52",
    "Others": "#4C72B0"
}


# ======================================================
# Helper
# ======================================================

def parse_protein(feature):
    return str(feature).replace("seq.", "")


def p_to_star(p):

    if pd.isna(p):
        return "n.s."
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "n.s."


def add_sig_for_each_protein(
    ax,
    plot_df,
    protein_order,
    x_col="abs_ig",
    y_col="protein",
    group_col="risk_group",
    group1="High risk",
    group2="Others",
    max_x=0.0025
):

    """
    For each protein, compare High risk vs Others using Mann–Whitney U test.
    Add significance bracket and stars to each horizontal boxplot row.
    """

    for i, protein in enumerate(protein_order):

        sub = plot_df[plot_df[y_col] == protein].copy()

        vals1 = sub.loc[sub[group_col] == group1, x_col].dropna().values
        vals2 = sub.loc[sub[group_col] == group2, x_col].dropna().values

        if len(vals1) < 2 or len(vals2) < 2:
            pval = np.nan
        else:
            _, pval = mannwhitneyu(
                vals1,
                vals2,
                alternative="two-sided"
            )

        star = p_to_star(pval)

        local_max = sub[x_col].max()

        if pd.isna(local_max):
            local_max = 0

        # significance bracket position
        x0 = min(local_max + max_x * 0.035, max_x * 0.84)
        x1 = min(local_max + max_x * 0.075, max_x * 0.89)
        x_text = min(local_max + max_x * 0.090, max_x * 0.925)

        # For horizontal boxplot with hue dodge,
        # the two boxes are approximately at y +/- 0.20
        y1 = i - 0.20
        y2 = i + 0.20
        y_mid = i

        ax.plot(
            [x0, x1, x1, x0],
            [y1, y1, y2, y2],
            color="black",
            linewidth=0.8,
            clip_on=False
        )

        ax.text(
            x_text,
            y_mid,
            star,
            va="center",
            ha="left",
            fontsize=10,
            color="black"
        )


# ======================================================
# Load data
# ======================================================

def load_data(ig_csv, pred_csv):

    ig = pd.read_csv(ig_csv)
    pred = pd.read_csv(pred_csv)

    pred["pid"] = pred["ID"]

    if "ig_abs" in ig.columns:
        ig["abs_ig"] = ig["ig_abs"].abs()
    else:
        ig["abs_ig"] = ig["ig"].abs()

    ig["protein"] = ig["feature"].apply(parse_protein)

    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)

    pred["risk_group"] = np.where(
        pred["risk"] >= risk_cut,
        "High risk",
        "Others"
    )

    ig = ig.merge(
        pred[["pid", "risk_group"]],
        on="pid",
        how="inner"
    )

    return ig


# ======================================================
# Top proteins
# ======================================================

def compute_top_proteins(ig):

    protein_sample = (
        ig
        .groupby(["pid", "protein", "risk_group"], as_index=False)["abs_ig"]
        .max()
    )

    protein_rank = (
        protein_sample[protein_sample["risk_group"] == "High risk"]
        .groupby("protein", as_index=False)["abs_ig"]
        .median()
        .sort_values("abs_ig", ascending=False)
    )

    top_proteins = protein_rank["protein"].head(TOP_N).tolist()

    plot_df = protein_sample[
        protein_sample["protein"].isin(top_proteins)
    ].copy()

    protein_order = (
        protein_rank
        .set_index("protein")
        .loc[top_proteins]
        .sort_values("abs_ig", ascending=False)
        .index
        .tolist()
    )

    return plot_df, protein_order


# ======================================================
# Draw boxplot
# ======================================================

def draw_boxplot(plot_df, protein_order, title, outfile):

    fig_h = 0.4 * len(protein_order) + 1

    fig, ax = plt.subplots(
        figsize=(4.2, 4.8)
    )

    sns.boxplot(
        data=plot_df,
        x="abs_ig",
        y="protein",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=protein_order,
        orient="h",
        showfliers=False,
        palette=BOX_PALETTE,
        linewidth=1,
        width=0.6,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="abs_ig",
        y="protein",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=protein_order,
        orient="h",
        dodge=True,
        jitter=0.25,
        size=3,
        alpha=0.9,
        edgecolor="black",
        linewidth=0.3,
        palette=POINT_PALETTE,
        ax=ax
    )

    # remove duplicated legends
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    # x axis name
    ax.set_xlabel("IG")
    ax.set_ylabel("")

    # optional title
    # ax.set_title(title, fontsize=11)

    # add significance
    add_sig_for_each_protein(
        ax=ax,
        plot_df=plot_df,
        protein_order=protein_order,
        x_col="abs_ig",
        y_col="protein",
        group_col="risk_group",
        group1="High risk",
        group2="Others",
        max_x=MAX_X
    )
    if "Coronary" in title:
        ax.set_xlim(0, 0.0035)
    else:
        ax.set_xlim(0, 0.002)

    # clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    plt.savefig(
        outfile,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", outfile)


# ======================================================
# Run CHD
# ======================================================

ig_chd = load_data(
    SOMA_IG_CSV_CHD,
    PRED_CSV_CHD
)

plot_chd, order_chd = compute_top_proteins(
    ig_chd
)

draw_boxplot(
    plot_chd,
    order_chd,
    "Coronary Heart Disease",
    OUT_CHD
)


# ======================================================
# Run iStroke
# ======================================================

ig_ist = load_data(
    SOMA_IG_CSV_IST,
    PRED_CSV_IST
)

plot_ist, order_ist = compute_top_proteins(
    ig_ist
)

draw_boxplot(
    plot_ist,
    order_ist,
    "Ischemic Stroke",
    OUT_IST
)

print("All done.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
# ======================================================
# Config
# ======================================================

ENDPOINTS = ["CHD", "istroke"]

SNP_IG_TEMPLATE = "code_02_02_save/snp_20260202_1432/explanations/{}_snp_TR_best_IG.csv"
PRED_TEMPLATE   = "code_02_02_save/snp_20260202_1432/preds/{}_snp_TR_test_best_pred.csv"

SNP_MAP = "fig2/SNP/CHD_IG_final.csv"

OUTDIR = "fig2_final"
os.makedirs(OUTDIR, exist_ok=True)

HIGH_RISK_Q = 0.85
TOP_N_LOCI = 10


MAX_X = 0.30

BOX_PALETTE = {
    "High risk": "#F3B0B2",
    "Others": "#AFC7E8"
}

POINT_PALETTE = {
    "High risk": "#C44E52",
    "Others": "#4C72B0"
}

sns.set_style("white")


# ======================================================
# Resolve gene annotation
# ======================================================

def resolve_gene(gene_list):

    if pd.isna(gene_list):
        return None

    genes = [g.strip() for g in str(gene_list).split(";")]

    # priority genes
    if "APOC3" in genes:
        return "APOC3"

    if "RNF215" in genes:
        return "RNF215"

    # remove ENSG
    genes = [g for g in genes if not g.startswith("ENSG")]

    if len(genes) == 0:
        return None

    return genes[0]


# ======================================================
# p value -> significance stars
# ======================================================

def p_to_star(p):

    if pd.isna(p):
        return "n.s."
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "n.s."


# ======================================================
# Add significance annotation for horizontal grouped boxplot
# ======================================================

def add_sig_for_each_locus(
    ax,
    plot_df,
    loci_order,
    x_col="abs_ig",
    y_col="label",
    group_col="risk_group",
    group1="High risk",
    group2="Others",
    max_x=0.30
):

    """Analysis helper. Inputs and outputs follow the function implementation."""

    for i, label in enumerate(loci_order):

        sub = plot_df[plot_df[y_col] == label].copy()

        vals1 = sub.loc[sub[group_col] == group1, x_col].dropna().values
        vals2 = sub.loc[sub[group_col] == group2, x_col].dropna().values

        if len(vals1) < 2 or len(vals2) < 2:
            pval = np.nan
        else:
            _, pval = mannwhitneyu(vals1, vals2, alternative="two-sided")

        star = p_to_star(pval)

        
        local_max = sub[x_col].max()

        if pd.isna(local_max):
            local_max = 0

        
        
        x0 = min(local_max + max_x * 0.035, max_x * 0.86)
        x1 = min(local_max + max_x * 0.075, max_x * 0.91)
        x_text = min(local_max + max_x * 0.085, max_x * 0.935)

        
        y1 = i - 0.20
        y2 = i + 0.20
        y_mid = i

        
        ax.plot(
            [x0, x1, x1, x0],
            [y1, y1, y2, y2],
            color="black",
            linewidth=0.8,
            clip_on=False
        )

        
        ax.text(
            x_text,
            y_mid,
            star,
            va="center",
            ha="left",
            fontsize=10,
            color="black"
        )


# ======================================================
# Load SNP → gene map
# ======================================================

snp_map = pd.read_csv(SNP_MAP)[["SNP", "gene"]]

# resolve duplicated gene annotations
snp_map["gene"] = snp_map["gene"].apply(resolve_gene)

# remove empty
snp_map = snp_map.dropna(subset=["gene"])


snp_map = snp_map.drop_duplicates(subset=["SNP"])


# ======================================================
# Main loop
# ======================================================

for ENDPOINT in ENDPOINTS:

    print("====================================")
    print("Running:", ENDPOINT)
    print("====================================")

    SNP_IG_CSV = SNP_IG_TEMPLATE.format(ENDPOINT)
    PRED_CSV   = PRED_TEMPLATE.format(ENDPOINT)

    ig = pd.read_csv(SNP_IG_CSV)
    pred = pd.read_csv(PRED_CSV)

    pred["pid"] = pred["ID"]

    ig = ig.merge(
        snp_map,
        left_on="feature",
        right_on="SNP",
        how="inner"
    )

    # abs IG
    if "ig_abs" in ig.columns:
        ig["abs_ig"] = ig["ig_abs"].abs()
    else:
        ig["abs_ig"] = ig["ig"].abs()

    # ======================================================
    # risk group
    # ======================================================

    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)

    pred["risk_group"] = np.where(
        pred["risk"] >= risk_cut,
        "High risk",
        "Others"
    )

    ig = ig.merge(
        pred[["pid", "risk_group"]],
        on="pid",
        how="inner"
    )

    # ======================================================
    # loci ranking
    # ======================================================

    loci_rank = (
        ig[ig["risk_group"] == "High risk"]
        .groupby(["feature", "gene"], as_index=False)["abs_ig"]
        .median()
        .sort_values("abs_ig", ascending=False)
    )

    # only rs loci
    loci_rank = loci_rank[
        loci_rank["feature"].astype(str).str.startswith("rs")
    ]

    # ======================================================
    # select top loci
    # ======================================================

    top_loci = loci_rank.head(TOP_N_LOCI).copy()

    top_loci["label"] = (
        top_loci["feature"]
        + " ("
        + top_loci["gene"]
        + ")"
    )

    loci_plot = ig.merge(
        top_loci[["feature", "label"]],
        on="feature",
        how="inner"
    )

    loci_order = (
        top_loci
        .sort_values("abs_ig", ascending=False)["label"]
        .tolist()
    )

    # ======================================================
    # Plot
    # ======================================================

    plt.figure(figsize=(3.6, 0.4 * len(loci_order) + 1))

    ax = sns.boxplot(
        data=loci_plot,
        x="abs_ig",
        y="label",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=loci_order,
        orient="h",
        showfliers=False,
        palette=BOX_PALETTE,
        linewidth=1
    )

    sns.stripplot(
        data=loci_plot,
        x="abs_ig",
        y="label",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=loci_order,
        orient="h",
        dodge=True,
        jitter=0.25,
        size=3,
        alpha=0.9,
        edgecolor="black",
        linewidth=0.3,
        palette=POINT_PALETTE
    )

    
    if ax.legend_ is not None:
        ax.legend_.remove()

    # ======================================================
    
    # ======================================================

    ax.set_xlabel("IG")
    ax.set_ylabel("")

    # ======================================================
    
    # ======================================================

    add_sig_for_each_locus(
        ax=ax,
        plot_df=loci_plot,
        loci_order=loci_order,
        x_col="abs_ig",
        y_col="label",
        group_col="risk_group",
        group1="High risk",
        group2="Others",
        max_x=MAX_X
    )

    
    ax.set_xlim(0, MAX_X)

    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    plt.savefig(
        f"{OUTDIR}/{ENDPOINT}_top_loci_origin_boxplot_with_sig.pdf",
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Loci boxplot with significance saved")

print("All done.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ======================================================
# Config
# ======================================================

TARGET_SNP = "rs1327115596"
WINDOW_BP = 5000
ENDPOINT = "CHD"

IG_CSV = f"code_02_02_save/snp_20260202_1432/explanations/{ENDPOINT}_snp_TR_best_IG.csv"
PRED_CSV = f"code_02_02_save/snp_20260202_1432/preds/{ENDPOINT}_snp_TR_test_best_pred.csv"

VARIANT_CSV = "data/private/CHD_SNP_variant_gene_with_symbol.csv"
GWAS_CATALOG = "data/private/gwas-catalog-download-associations-v1.0-full.tsv"

HIGH_RISK_Q = 0.85
BLOCK_WIDTH = 12
# ======================================================

# ======================================================

ALLOWED_JOURNALS = [
    "Nature",
    "Nat Commun",
    "Nat Genet",
    "J Am Heart Assoc",
]

# ======================================================


# ======================================================

MANUAL_SELECTED_TRACKS = [
    {
        "DISEASE/TRAIT": "Total cholesterol levels in medium HDL",
        "FIRST_AUTHOR_CLEAN": "Karjalainen",
        "YEAR": "2024",
        "JOURNAL": "Nature",
    },
    {
        "DISEASE/TRAIT": "Triglyceride levels",
        "FIRST_AUTHOR_CLEAN": "Sohail",
        "YEAR": "2023",
        "JOURNAL": "Nature",
    },
    {
        "DISEASE/TRAIT": "High-density lipoprotein levels (MTAG)",
        "FIRST_AUTHOR_CLEAN": "Koskeridis",
        "YEAR": "2022",
        "JOURNAL": "Nat Commun",
    },
    {
        "DISEASE/TRAIT": "Non-HDL cholesterol levels",
        "FIRST_AUTHOR_CLEAN": "Jacobs",
        "YEAR": "2024",
        "JOURNAL": "Nat Commun",
    },
    {
        "DISEASE/TRAIT": "High density lipoprotein cholesterol levels",
        "FIRST_AUTHOR_CLEAN": "Sinnott-Armstrong",
        "YEAR": "2021",
        "JOURNAL": "Nat Genet",
    },
]


HALF_WINDOW = 500  # 0.5 kb

OUTDIR = "fig2_final/locus_plots"
os.makedirs(OUTDIR, exist_ok=True)

OUTFIG = f"{OUTDIR}/{TARGET_SNP}_{ENDPOINT}_locus_pretty_manual_5tracks.pdf"
OUT_SELECTED = f"{OUTDIR}/{TARGET_SNP}_{ENDPOINT}_selected_GWAS_tracks_manual_5tracks.csv"

sns.set_style("white")

# ======================================================

# ======================================================

TRAIT_COLORS = [
    "#6EB8D2",
    "#479D88",
    "#435385",
    "#E6A083",
    "#8790B2",
]

# ======================================================
# Helper functions
# ======================================================

def parse_variant(v):
    """
    Parse variant_id like chr1_12345_A_G.
    """
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    return np.nan, np.nan


def clean_author(author):
    """
    Convert 'Sinnott-Armstrong N' -> 'Sinnott-Armstrong'
    Convert 'Koskeridis F' -> 'Koskeridis'
    """
    author = str(author).strip()
    if author == "" or author.lower() == "nan":
        return "NA"

    parts = author.split()

    
    if len(parts) >= 2 and re.fullmatch(r"[A-Z]+\.?", parts[-1]):
        return " ".join(parts[:-1])

    return author


def get_year(date_value):
    """
    Extract year from DATE column.
    """
    dt = pd.to_datetime(date_value, errors="coerce")
    if pd.isna(dt):
        return "NA"
    return str(dt.year)


def add_source_columns(df):
    """
    Add source columns used for selection and legend labels.
    """
    df = df.copy()

    df["JOURNAL"] = df["JOURNAL"].astype(str).str.strip()
    df["DISEASE/TRAIT"] = df["DISEASE/TRAIT"].astype(str).str.strip()

    df["FIRST_AUTHOR_CLEAN"] = df["FIRST AUTHOR"].apply(clean_author)
    df["YEAR"] = df["DATE"].apply(get_year)

    df["SOURCE_LABEL"] = (
        df["FIRST_AUTHOR_CLEAN"].astype(str)
        + ". "
        + df["YEAR"].astype(str)
        + ". "
        + df["JOURNAL"].astype(str)
    )

    
    df["TRACK_ID"] = (
        df["DISEASE/TRAIT"].astype(str)
        + "||"
        + df["FIRST_AUTHOR_CLEAN"].astype(str)
        + "||"
        + df["YEAR"].astype(str)
        + "||"
        + df["JOURNAL"].astype(str)
    )

    df["LEGEND_LABEL"] = (
        df["DISEASE/TRAIT"].astype(str)
        + " ("
        + df["SOURCE_LABEL"]
        + ")"
    )

    return df


def build_manual_track_id(item):
    """
    Build TRACK_ID from manual selection dictionary.
    """
    return (
        str(item["DISEASE/TRAIT"]).strip()
        + "||"
        + str(item["FIRST_AUTHOR_CLEAN"]).strip()
        + "||"
        + str(item["YEAR"]).strip()
        + "||"
        + str(item["JOURNAL"]).strip()
    )


def select_manual_gwas_tracks(gwas_region, manual_selected_tracks, allowed_journals):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    gwas_region = gwas_region.copy()
    allowed_journals = [j.strip() for j in allowed_journals]

    
    gwas_region = gwas_region[
        gwas_region["JOURNAL"].isin(allowed_journals)
    ].copy()

    if gwas_region.empty:
        raise ValueError(
            "No GWAS associations found after filtering by ALLOWED_JOURNALS. "
            f"Allowed journals: {allowed_journals}"
        )

    manual_track_ids = [
        build_manual_track_id(x)
        for x in manual_selected_tracks
    ]

    gwas_region = gwas_region[
        gwas_region["TRACK_ID"].isin(manual_track_ids)
    ].copy()

    if gwas_region.empty:
        raise ValueError(
            "No GWAS associations found for MANUAL_SELECTED_TRACKS. "
            "Please check trait names, author names, year, and journal."
        )

    
    gwas_region["P-VALUE"] = pd.to_numeric(gwas_region["P-VALUE"], errors="coerce")
    gwas_region["CHR_POS"] = pd.to_numeric(gwas_region["CHR_POS"], errors="coerce")
    gwas_region["PVALUE_MLOG"] = pd.to_numeric(gwas_region["PVALUE_MLOG"], errors="coerce")

    summary = (
        gwas_region
        .groupby(
            [
                "TRACK_ID",
                "DISEASE/TRAIT",
                "FIRST_AUTHOR_CLEAN",
                "YEAR",
                "JOURNAL",
                "SOURCE_LABEL",
                "LEGEND_LABEL",
            ],
            dropna=False,
            as_index=False
        )
        .agg(
            n_assoc=("TRACK_ID", "size"),
            n_sites=("CHR_POS", "nunique"),
            min_p=("P-VALUE", "min"),
            max_mlogp=("PVALUE_MLOG", "max")
        )
    )

    
    order_map = {
        track_id: i
        for i, track_id in enumerate(manual_track_ids)
    }

    summary["manual_order"] = summary["TRACK_ID"].map(order_map)
    summary = summary.sort_values("manual_order").drop(columns=["manual_order"])
    summary = summary.reset_index(drop=True)

    
    found_track_ids = set(summary["TRACK_ID"])
    missing_track_ids = [
        x for x in manual_track_ids
        if x not in found_track_ids
    ]

    if len(missing_track_ids) > 0:
        print("\n[Warning] These manual tracks were not found in GWAS region:")
        for x in missing_track_ids:
            print("  -", x)

    return summary, gwas_region


# ======================================================
# Load variant annotation
# ======================================================

variant = pd.read_csv(VARIANT_CSV)

variant[["chr", "pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant(x))
)

variant = variant.dropna(subset=["chr", "pos"]).copy()
variant["pos"] = variant["pos"].astype(int)

# ======================================================
# Target SNP
# ======================================================

target = variant[variant["SNP"] == TARGET_SNP].copy()

if target.empty:
    raise ValueError(f"TARGET_SNP not found in VARIANT_CSV: {TARGET_SNP}")

target_chr = target["chr"].iloc[0]
target_pos = int(target["pos"].iloc[0])

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

print("Target SNP:", TARGET_SNP)
print("Target chr:", target_chr)
print("Target pos:", target_pos)
print("Window:", left, "-", right)

# ======================================================
# Load IG and prediction
# ======================================================

ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)

pred["pid"] = pred["ID"]

ig = ig.merge(
    variant[["SNP", "chr", "pos"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)

ig["abs_ig"] = ig["ig_abs"].abs()

# high risk filtering
cut = pred["risk"].quantile(HIGH_RISK_Q)
high_pid = pred.loc[pred["risk"] >= cut, "pid"]

ig = ig[ig["pid"].isin(high_pid)].copy()

# locus subset
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
].copy()

if locus.empty:
    raise ValueError("No IG SNPs found in this locus after high-risk filtering.")

# aggregate IG
snp_agg = (
    locus
    .groupby(["SNP", "pos"], as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig": "mean_abs_ig"})
)

snp_agg["rel_pos"] = snp_agg["pos"] - target_pos

# ======================================================
# Load GWAS catalog
# ======================================================

gwas = pd.read_csv(GWAS_CATALOG, sep="\t", low_memory=False)


gwas.columns = gwas.columns.str.strip()

required_cols = [
    "CHR_ID",
    "CHR_POS",
    "DISEASE/TRAIT",
    "FIRST AUTHOR",
    "DATE",
    "JOURNAL",
    "P-VALUE",
    "PVALUE_MLOG"
]

missing_cols = [c for c in required_cols if c not in gwas.columns]
if missing_cols:
    raise ValueError(f"Missing columns in GWAS catalog: {missing_cols}")

gwas["CHR_ID"] = pd.to_numeric(gwas["CHR_ID"], errors="coerce")
gwas["CHR_POS"] = pd.to_numeric(gwas["CHR_POS"], errors="coerce")
gwas["P-VALUE"] = pd.to_numeric(gwas["P-VALUE"], errors="coerce")
gwas["PVALUE_MLOG"] = pd.to_numeric(gwas["PVALUE_MLOG"], errors="coerce")

chr_numeric = int(target_chr.replace("chr", ""))

gwas_region = gwas[
    (gwas["CHR_ID"] == chr_numeric) &
    (gwas["CHR_POS"] >= left) &
    (gwas["CHR_POS"] <= right)
].copy()

if gwas_region.empty:
    raise ValueError("No GWAS Catalog associations found in this locus.")

gwas_region = add_source_columns(gwas_region)
gwas_region["rel_pos"] = gwas_region["CHR_POS"] - target_pos

# ======================================================

# ======================================================

selected_tracks, gwas_region_selected = select_manual_gwas_tracks(
    gwas_region,
    manual_selected_tracks=MANUAL_SELECTED_TRACKS,
    allowed_journals=ALLOWED_JOURNALS
)

selected_track_ids = selected_tracks["TRACK_ID"].tolist()

gwas_region_plot = gwas_region_selected[
    gwas_region_selected["TRACK_ID"].isin(selected_track_ids)
].copy()


track_order = selected_tracks["TRACK_ID"].tolist()
legend_labels = selected_tracks["LEGEND_LABEL"].tolist()

track_to_y = {
    track_id: i
    for i, track_id in enumerate(track_order)
}

track_to_color = {
    track_id: TRAIT_COLORS[i % len(TRAIT_COLORS)]
    for i, track_id in enumerate(track_order)
}

gwas_region_plot["track_y"] = gwas_region_plot["TRACK_ID"].map(track_to_y)
gwas_region_plot["track_color"] = gwas_region_plot["TRACK_ID"].map(track_to_color)


selected_tracks.to_csv(OUT_SELECTED, index=False)

print("\nAllowed journals:")
print(ALLOWED_JOURNALS)

print("\nSelected GWAS tracks:")
print(
    selected_tracks[
        [
            "DISEASE/TRAIT",
            "SOURCE_LABEL",
            "JOURNAL",
            "n_sites",
            "n_assoc",
            "min_p"
        ]
    ].to_string(index=False)
)

print("\nSaved selected track table:", OUT_SELECTED)

# ======================================================
# Plot
# ======================================================
# ======================================================
# Plot
# ======================================================
# ======================================================
# Plot
# ======================================================
# ======================================================
# Plot
# ======================================================

n_tracks = len(track_order)

fig = plt.figure(figsize=(10, 4.6))

# ======================================================

# ======================================================
PANEL_LEFT = 0.10
PANEL_WIDTH = 0.72


TOP_BOTTOM = 0.62
TOP_HEIGHT = 0.12


BOTTOM_BOTTOM = 0.20
BOTTOM_HEIGHT = 0.26

TRACK_SPACING = 1.45   
BLOCK_WIDTH = 13       


track_to_y = {
    track_id: i * TRACK_SPACING
    for i, track_id in enumerate(track_order)
}

gwas_region_plot["track_y"] = gwas_region_plot["TRACK_ID"].map(track_to_y)

# ======================================================
# Upper locus panel
# ======================================================
ax = fig.add_axes([PANEL_LEFT, TOP_BOTTOM, PANEL_WIDTH, TOP_HEIGHT])

sns.scatterplot(
    data=snp_agg,
    x="rel_pos",
    y="mean_abs_ig",
    color="#4C72B0",
    edgecolor="black",
    linewidth=0.3,
    s=45,
    ax=ax
)

# SNP labels
for _, row in snp_agg.iterrows():
    ax.text(
        row["rel_pos"],
        row["mean_abs_ig"] + 0.0008,
        row["SNP"],
        fontsize=7,
        ha="center",
        va="bottom",
        clip_on=False
    )

# target SNP vertical line
ax.axvline(
    0,
    color="black",
    linewidth=0.8,
    linestyle="--",
    alpha=0.5
)


ax.set_xlim(-WINDOW_BP, WINDOW_BP)


ax.set_xticks([-WINDOW_BP, 0, WINDOW_BP])
ax.set_xticklabels(["-5k", "0", "5k"])

ax.set_ylabel("Mean IG")
ax.set_xlabel("")


ax.set_ylim(0.039, 0.051)
ax.set_yticks([0.040, 0.045, 0.050])
ax.set_yticklabels(["0.040", "0.045", "0.050"])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ======================================================
# Lower region panel
# ======================================================
ax_trait = fig.add_axes(
    [PANEL_LEFT, BOTTOM_BOTTOM, PANEL_WIDTH, BOTTOM_HEIGHT],
    sharex=ax
)

handles = []

for i, track_id in enumerate(track_order):
    df = gwas_region_plot[gwas_region_plot["TRACK_ID"] == track_id].copy()
    color = track_to_color[track_id]
    y_pos = track_to_y[track_id]

    for p in df["rel_pos"]:
        ax_trait.hlines(
            y=y_pos,
            xmin=p - HALF_WINDOW,
            xmax=p + HALF_WINDOW,
            color=color,
            linewidth=BLOCK_WIDTH
        )

    handles.append(
        plt.Line2D([0], [0], color=color, lw=6)
    )

# target SNP vertical line
ax_trait.axvline(
    0,
    color="black",
    linewidth=0.8,
    linestyle="--",
    alpha=0.5
)


ax_trait.set_xlim(-WINDOW_BP, WINDOW_BP)


ax_trait.set_xticks([-WINDOW_BP, 0, WINDOW_BP])
ax_trait.set_xticklabels(["-5k", "0", "5k"])


y_min = -0.8
y_max = (n_tracks - 1) * TRACK_SPACING + 0.8
ax_trait.set_ylim(y_min, y_max)

ax_trait.set_yticks([])
ax_trait.set_xlabel("Distance from target SNP (bp)")

for spine in ax_trait.spines.values():
    spine.set_visible(False)

# ======================================================
# Legend outside
# ======================================================
fig.legend(
    handles,
    legend_labels,
    loc="center left",
    bbox_to_anchor=(0.84, 0.52),
    frameon=False,
    title="GWAS traits",
    fontsize=8,
    title_fontsize=9
)

plt.savefig(
    OUTFIG,
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)

plt.show()

print("Saved figure:", OUTFIG)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ======================================================
# Config
# ======================================================

TARGET_SNP = "rs3782805"
WINDOW_BP = 6000
ENDPOINT = "istroke"

IG_CSV = f"code_02_02_save/snp_20260202_1432/explanations/{ENDPOINT}_snp_TR_best_IG.csv"
PRED_CSV = f"code_02_02_save/snp_20260202_1432/preds/{ENDPOINT}_snp_TR_test_best_pred.csv"

VARIANT_CSV = "data/private/CHD_SNP_variant_gene_with_symbol.csv"
GWAS_CATALOG = "data/private/gwas-catalog-download-associations-v1.0-full.tsv"

HIGH_RISK_Q = 0.85

# ======================================================

# ======================================================

MANUAL_SELECTED_TRACKS = [
    {
        "DISEASE/TRAIT": "Lung function (FVC)",
        "FIRST_AUTHOR_CLEAN": "Kichaev",
        "YEAR": "2018",
        "JOURNAL": "Am J Hum Genet",
    },
    {
        "DISEASE/TRAIT": "Physical function (baseline)",
        "FIRST_AUTHOR_CLEAN": "Schoeler",
        "YEAR": "2025",
        "JOURNAL": "Nat Commun",
    },
    {
        "DISEASE/TRAIT": "Height (baseline)",
        "FIRST_AUTHOR_CLEAN": "Schoeler",
        "YEAR": "2025",
        "JOURNAL": "Nat Commun",
    },
    {
        "DISEASE/TRAIT": "Forced expiratory volume (baseline)",
        "FIRST_AUTHOR_CLEAN": "Schoeler",
        "YEAR": "2025",
        "JOURNAL": "Nat Commun",
    },
    {
        "DISEASE/TRAIT": "White matter lesions (Fazekas score)",
        "FIRST_AUTHOR_CLEAN": "Homann",
        "YEAR": "2022",
        "JOURNAL": "Front Aging Neurosci",
    },
]

HALF_WINDOW = 500  

OUTDIR = "fig2_final/locus_plots"
os.makedirs(OUTDIR, exist_ok=True)

OUTFIG = f"{OUTDIR}/{TARGET_SNP}_{ENDPOINT}_locus_pretty_manual_5tracks.pdf"
OUT_SELECTED = f"{OUTDIR}/{TARGET_SNP}_{ENDPOINT}_selected_GWAS_tracks_manual_5tracks.csv"

sns.set_style("white")

# ======================================================
# Colors
# ======================================================

TRAIT_COLORS = [
    "#6EB8D2",
    "#479D88",
    "#435385",
    "#E6A083",
    "#8790B2",
]

# ======================================================
# Helper functions
# ======================================================

def parse_variant(v):
    """
    Parse variant_id like chr1_12345_A_G
    """
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    return np.nan, np.nan


def clean_author(author):
    """
    Convert 'Sinnott-Armstrong N' -> 'Sinnott-Armstrong'
    Convert 'Koskeridis F' -> 'Koskeridis'
    """
    author = str(author).strip()
    if author == "" or author.lower() == "nan":
        return "NA"

    parts = author.split()

    
    if len(parts) >= 2 and re.fullmatch(r"[A-Z]+\.?", parts[-1]):
        return " ".join(parts[:-1])

    return author


def get_year(date_value):
    """
    Extract year from DATE column
    """
    dt = pd.to_datetime(date_value, errors="coerce")
    if pd.isna(dt):
        return "NA"
    return str(dt.year)


def add_gwas_source_columns(df):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    df = df.copy()

    df["DISEASE/TRAIT"] = df["DISEASE/TRAIT"].astype(str).str.strip()
    df["JOURNAL"] = df["JOURNAL"].astype(str).str.strip()

    df["FIRST_AUTHOR_CLEAN"] = df["FIRST AUTHOR"].apply(clean_author)
    df["YEAR"] = df["DATE"].apply(get_year)

    df["SOURCE_LABEL"] = (
        df["FIRST_AUTHOR_CLEAN"].astype(str)
        + ". "
        + df["YEAR"].astype(str)
        + ". "
        + df["JOURNAL"].astype(str)
    )

    df["TRACK_ID"] = (
        df["DISEASE/TRAIT"].astype(str)
        + "||"
        + df["FIRST_AUTHOR_CLEAN"].astype(str)
        + "||"
        + df["YEAR"].astype(str)
        + "||"
        + df["JOURNAL"].astype(str)
    )

    df["LEGEND_LABEL"] = (
        df["DISEASE/TRAIT"].astype(str)
        + " ("
        + df["SOURCE_LABEL"].astype(str)
        + ")"
    )

    return df


def build_manual_track_id(item):
    """
    Build TRACK_ID from manual selection dictionary
    """
    return (
        str(item["DISEASE/TRAIT"]).strip()
        + "||"
        + str(item["FIRST_AUTHOR_CLEAN"]).strip()
        + "||"
        + str(item["YEAR"]).strip()
        + "||"
        + str(item["JOURNAL"]).strip()
    )


def select_manual_gwas_tracks(gwas_region, manual_selected_tracks):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    gwas_region = gwas_region.copy()

    manual_track_ids = [build_manual_track_id(x) for x in manual_selected_tracks]

    gwas_region = gwas_region[gwas_region["TRACK_ID"].isin(manual_track_ids)].copy()

    if gwas_region.empty:
        raise ValueError(
            "No GWAS associations found for MANUAL_SELECTED_TRACKS. "
            "Please check trait / author / year / journal."
        )

    gwas_region["P-VALUE"] = pd.to_numeric(gwas_region["P-VALUE"], errors="coerce")
    gwas_region["CHR_POS"] = pd.to_numeric(gwas_region["CHR_POS"], errors="coerce")
    gwas_region["PVALUE_MLOG"] = pd.to_numeric(gwas_region["PVALUE_MLOG"], errors="coerce")

    summary = (
        gwas_region
        .groupby(
            [
                "TRACK_ID",
                "DISEASE/TRAIT",
                "FIRST_AUTHOR_CLEAN",
                "YEAR",
                "JOURNAL",
                "SOURCE_LABEL",
                "LEGEND_LABEL",
            ],
            as_index=False,
            dropna=False
        )
        .agg(
            n_assoc=("TRACK_ID", "size"),
            n_sites=("CHR_POS", "nunique"),
            min_p=("P-VALUE", "min"),
            max_mlogp=("PVALUE_MLOG", "max")
        )
    )

    
    order_map = {
        build_manual_track_id(item): i
        for i, item in enumerate(manual_selected_tracks)
    }

    summary["manual_order"] = summary["TRACK_ID"].map(order_map)
    summary = summary.sort_values("manual_order").drop(columns=["manual_order"])
    summary = summary.reset_index(drop=True)

    
    found_track_ids = set(summary["TRACK_ID"])
    missing_track_ids = [x for x in manual_track_ids if x not in found_track_ids]

    if len(missing_track_ids) > 0:
        print("\n[Warning] These manual tracks were not found:")
        for x in missing_track_ids:
            print("  -", x)

    return summary, gwas_region


# ======================================================
# Load variant annotation
# ======================================================

variant = pd.read_csv(VARIANT_CSV)

variant[["chr", "pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant(x))
)

variant = variant.dropna(subset=["chr", "pos"]).copy()
variant["pos"] = variant["pos"].astype(int)

# ======================================================
# Target SNP
# ======================================================

target = variant[variant["SNP"] == TARGET_SNP].copy()

if target.empty:
    raise ValueError(f"TARGET_SNP not found in VARIANT_CSV: {TARGET_SNP}")

target_chr = target["chr"].iloc[0]
target_pos = int(target["pos"].iloc[0])

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

print("Target SNP:", TARGET_SNP)
print("Target chr:", target_chr)
print("Target pos:", target_pos)
print("Window:", left, "-", right)

# ======================================================
# Load IG
# ======================================================

ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)

pred["pid"] = pred["ID"]

ig = ig.merge(
    variant[["SNP", "chr", "pos"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)

ig["abs_ig"] = ig["ig_abs"].abs()

# high risk filtering
cut = pred["risk"].quantile(HIGH_RISK_Q)
high_pid = pred.loc[pred["risk"] >= cut, "pid"]

ig = ig[ig["pid"].isin(high_pid)].copy()

# locus subset
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
].copy()

if locus.empty:
    raise ValueError("No IG SNPs found in this locus after high-risk filtering.")

# aggregate IG
snp_agg = (
    locus
    .groupby(["SNP", "pos"], as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig": "mean_abs_ig"})
)

snp_agg["rel_pos"] = snp_agg["pos"] - target_pos

# ======================================================
# Load GWAS catalog
# ======================================================

gwas = pd.read_csv(GWAS_CATALOG, sep="\t", low_memory=False)
gwas.columns = gwas.columns.str.strip()

required_cols = [
    "CHR_ID",
    "CHR_POS",
    "DISEASE/TRAIT",
    "FIRST AUTHOR",
    "DATE",
    "JOURNAL",
    "P-VALUE",
    "PVALUE_MLOG",
]

missing_cols = [c for c in required_cols if c not in gwas.columns]
if len(missing_cols) > 0:
    raise ValueError(f"Missing columns in GWAS catalog: {missing_cols}")

gwas["CHR_ID"] = pd.to_numeric(gwas["CHR_ID"], errors="coerce")
gwas["CHR_POS"] = pd.to_numeric(gwas["CHR_POS"], errors="coerce")
gwas["P-VALUE"] = pd.to_numeric(gwas["P-VALUE"], errors="coerce")
gwas["PVALUE_MLOG"] = pd.to_numeric(gwas["PVALUE_MLOG"], errors="coerce")

chr_numeric = int(target_chr.replace("chr", ""))

gwas_region = gwas[
    (gwas["CHR_ID"] == chr_numeric) &
    (gwas["CHR_POS"] >= left) &
    (gwas["CHR_POS"] <= right)
].copy()

if gwas_region.empty:
    raise ValueError("No GWAS Catalog associations found in this locus.")

gwas_region = add_gwas_source_columns(gwas_region)
gwas_region["rel_pos"] = gwas_region["CHR_POS"] - target_pos

# ======================================================

# ======================================================

selected_tracks, gwas_region_selected = select_manual_gwas_tracks(
    gwas_region,
    manual_selected_tracks=MANUAL_SELECTED_TRACKS
)

selected_track_ids = selected_tracks["TRACK_ID"].tolist()

gwas_region_plot = gwas_region_selected[
    gwas_region_selected["TRACK_ID"].isin(selected_track_ids)
].copy()

track_order = selected_tracks["TRACK_ID"].tolist()
legend_labels = selected_tracks["LEGEND_LABEL"].tolist()

track_to_y = {track_id: i for i, track_id in enumerate(track_order)}
track_to_color = {
    track_id: TRAIT_COLORS[i % len(TRAIT_COLORS)]
    for i, track_id in enumerate(track_order)
}

gwas_region_plot["track_y"] = gwas_region_plot["TRACK_ID"].map(track_to_y)
gwas_region_plot["track_color"] = gwas_region_plot["TRACK_ID"].map(track_to_color)

selected_tracks.to_csv(OUT_SELECTED, index=False)

print("\nSelected GWAS tracks:")
print(
    selected_tracks[
        [
            "DISEASE/TRAIT",
            "SOURCE_LABEL",
            "JOURNAL",
            "n_sites",
            "n_assoc",
            "min_p"
        ]
    ].to_string(index=False)
)

print("\nSaved selected GWAS table:", OUT_SELECTED)

# ======================================================
# Plot
# ======================================================
# ======================================================
# Plot
# ======================================================

n_tracks = len(track_order)

fig = plt.figure(figsize=(10, 4.6))

# ======================================================

# ======================================================
PANEL_LEFT = 0.10
PANEL_WIDTH = 0.72


TOP_BOTTOM = 0.62
TOP_HEIGHT = 0.12


BOTTOM_BOTTOM = 0.20
BOTTOM_HEIGHT = 0.26

TRACK_SPACING = 1.45   
BLOCK_WIDTH = 13       


track_to_y = {
    track_id: i * TRACK_SPACING
    for i, track_id in enumerate(track_order)
}

gwas_region_plot["track_y"] = gwas_region_plot["TRACK_ID"].map(track_to_y)

# ======================================================
# Upper locus panel
# ======================================================
ax = fig.add_axes([PANEL_LEFT, TOP_BOTTOM, PANEL_WIDTH, TOP_HEIGHT])

sns.scatterplot(
    data=snp_agg,
    x="rel_pos",
    y="mean_abs_ig",
    color="#4C72B0",
    edgecolor="black",
    linewidth=0.3,
    s=45,
    ax=ax
)

# SNP labels
for _, row in snp_agg.iterrows():
    ax.text(
        row["rel_pos"],
        row["mean_abs_ig"] + 0.0008,
        row["SNP"],
        fontsize=7,
        ha="center",
        va="bottom",
        clip_on=False
    )

# target SNP vertical line
ax.axvline(
    0,
    color="black",
    linewidth=0.8,
    linestyle="--",
    alpha=0.5
)


ax.set_xlim(-WINDOW_BP, WINDOW_BP)


ax.set_xticks([-WINDOW_BP, 0, WINDOW_BP])
ax.set_xticklabels(["-5k", "0", "5k"])

ax.set_ylabel("Mean IG")
ax.set_xlabel("")


ax.set_ylim(0.044, 0.056)
ax.set_yticks([0.045, 0.050, 0.055])
ax.set_yticklabels(["0.045", "0.050", "0.055"])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ======================================================
# Lower region panel
# ======================================================
ax_trait = fig.add_axes(
    [PANEL_LEFT, BOTTOM_BOTTOM, PANEL_WIDTH, BOTTOM_HEIGHT],
    sharex=ax
)

handles = []

for i, track_id in enumerate(track_order):
    df = gwas_region_plot[gwas_region_plot["TRACK_ID"] == track_id].copy()
    color = track_to_color[track_id]
    y_pos = track_to_y[track_id]

    for p in df["rel_pos"]:
        ax_trait.hlines(
            y=y_pos,
            xmin=p - HALF_WINDOW,
            xmax=p + HALF_WINDOW,
            color=color,
            linewidth=BLOCK_WIDTH
        )

    handles.append(
        plt.Line2D([0], [0], color=color, lw=6)
    )

# target SNP vertical line
ax_trait.axvline(
    0,
    color="black",
    linewidth=0.8,
    linestyle="--",
    alpha=0.5
)


ax_trait.set_xlim(-WINDOW_BP, WINDOW_BP)


ax_trait.set_xticks([-WINDOW_BP, 0, WINDOW_BP])
ax_trait.set_xticklabels(["-5k", "0", "5k"])


y_min = -0.8
y_max = (n_tracks - 1) * TRACK_SPACING + 0.8
ax_trait.set_ylim(y_min, y_max)

ax_trait.set_yticks([])
ax_trait.set_xlabel("Distance from target SNP (bp)")

for spine in ax_trait.spines.values():
    spine.set_visible(False)

# ======================================================
# Legend outside
# ======================================================
fig.legend(
    handles,
    legend_labels,
    loc="center left",
    bbox_to_anchor=(0.84, 0.52),
    frameon=False,
    title="GWAS traits",
    fontsize=8,
    title_fontsize=9
)

plt.savefig(
    OUTFIG,
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)

plt.show()

print("Saved figure:", OUTFIG)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ======================================================
# Config
# ======================================================

TARGET_SNP = "rs3782805"
WINDOW_BP = 5000
ENDPOINT = "istroke"

IG_CSV = f"code_02_02_save/snp_20260202_1432/explanations/{ENDPOINT}_snp_TR_best_IG.csv"
PRED_CSV = f"code_02_02_save/snp_20260202_1432/preds/{ENDPOINT}_snp_TR_test_best_pred.csv"

VARIANT_CSV = "data/private/CHD_SNP_variant_gene_with_symbol.csv"
GWAS_CATALOG = "data/private/gwas-catalog-download-associations-v1.0-full.tsv"

HIGH_RISK_Q = 0.85
MAX_TRAITS = 15

OUTDIR = "fig2_final/locus_plots"
os.makedirs(OUTDIR, exist_ok=True)

OUTFIG = f"{OUTDIR}/{TARGET_SNP}_{ENDPOINT}_locus_pretty.pdf"

sns.set_style("white")

# ======================================================
# Beautiful palette
# ======================================================

TRAIT_COLORS = [
    "#6EB8D2",  # deep red
    "#479D88",  # slate blue
    "#435385",  # teal green
    "#E6A083",  # muted purple
    "#8790B2"   # warm gold
]
# ======================================================
# Parse variant positions
# ======================================================

variant = pd.read_csv(VARIANT_CSV)

def parse_variant(v):

    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))

    if m:
        return m.group(1), int(m.group(2))

    return np.nan, np.nan

variant[["chr","pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant(x))
)

variant = variant.dropna(subset=["chr","pos"])

# ======================================================
# Target SNP
# ======================================================

target = variant[variant["SNP"] == TARGET_SNP]

target_chr = target["chr"].iloc[0]
target_pos = target["pos"].iloc[0]

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

# ======================================================
# Load IG
# ======================================================

ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)

pred["pid"] = pred["ID"]

ig = ig.merge(
    variant[["SNP","chr","pos"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)

ig["abs_ig"] = ig["ig_abs"].abs()

# high risk filtering
cut = pred["risk"].quantile(HIGH_RISK_Q)
high_pid = pred.loc[pred["risk"] >= cut, "pid"]

ig = ig[ig["pid"].isin(high_pid)]

# locus subset
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
]

# aggregate
snp_agg = (
    locus
    .groupby(["SNP","pos"],as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig":"mean_abs_ig"})
)

# relative position
snp_agg["rel_pos"] = snp_agg["pos"] - target_pos

# ======================================================
# Load GWAS catalog
# ======================================================

gwas = pd.read_csv(GWAS_CATALOG, sep="\t", low_memory=False)

gwas["CHR_ID"] = pd.to_numeric(gwas["CHR_ID"], errors="coerce")
gwas["CHR_POS"] = pd.to_numeric(gwas["CHR_POS"], errors="coerce")

chr_numeric = int(target_chr.replace("chr",""))

gwas_region = gwas[
    (gwas["CHR_ID"] == chr_numeric) &
    (gwas["CHR_POS"] >= left) &
    (gwas["CHR_POS"] <= right)
]

# choose top traits
trait_counts = gwas_region["DISEASE/TRAIT"].value_counts()

top_traits = trait_counts.head(MAX_TRAITS).index.tolist()

gwas_region = gwas_region[
    gwas_region["DISEASE/TRAIT"].isin(top_traits)
]

gwas_region["rel_pos"] = gwas_region["CHR_POS"] - target_pos

# ======================================================
# Plot
# ======================================================

fig = plt.figure(figsize=(10,5))

# IG axis
ax = fig.add_axes([0.1,0.4,0.72,0.25])

sns.scatterplot(
    data=snp_agg,
    x="rel_pos",
    y="mean_abs_ig",
    color="#4C72B0",
    edgecolor="black",
    linewidth=0.3,
    s=45,
    ax=ax
)

# SNP labels
for _,row in snp_agg.iterrows():

    ax.text(
        row["rel_pos"],
        row["mean_abs_ig"] + 0.0005,
        row["SNP"],
        fontsize=7,
        ha="center"
    )

ax.set_ylabel("Mean IG")
ax.set_xlabel("")
ax.set_xlim(-WINDOW_BP, WINDOW_BP)

# ======================================================
# Trait track (bars)
# ======================================================

ax_trait = fig.add_axes([0.1,0.22,0.72,0.12], sharex=ax)


handles = []

HALF_WINDOW = 500  # 0.5 kb

for i,trait in enumerate(top_traits):

    df = gwas_region[
        gwas_region["DISEASE/TRAIT"] == trait
    ]

    # draw a small bar around each SNP
    for p in df["rel_pos"]:

        ax_trait.hlines(
            y=i,
            xmin=p - HALF_WINDOW,
            xmax=p + HALF_WINDOW,
            color=TRAIT_COLORS[i],
            linewidth=8
        )

    handles.append(
        plt.Line2D([0],[0],color=TRAIT_COLORS[i],lw=6)
    )

ax_trait.set_xlim(-WINDOW_BP, WINDOW_BP)
# ticks every 1 kb
ticks = np.arange(-WINDOW_BP, WINDOW_BP + 1, 5000)

ax_trait.set_xticks(ticks)

ax_trait.set_xticklabels(
    [f"{int(t/1000)}k" if t != 0 else "0" for t in ticks]
)
ax_trait.set_yticks([])

# x axis label
ax_trait.set_xlabel("Distance from target SNP (bp)")

for spine in ax_trait.spines.values():
    spine.set_visible(False)

# ======================================================
# Legend outside
# ======================================================

fig.legend(
    handles,
    top_traits,
    loc="center left",
    bbox_to_anchor=(0.85,0.5),
    frameon=False,
    title="GWAS traits"
)
'''
# ======================================================
# Title
# ======================================================

plt.suptitle(
    f"{TARGET_SNP} locus ±{WINDOW_BP} bp (Ischemic Stroke)",
    y=1.10
)
'''
plt.savefig(
    OUTFIG,
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)

plt.show()

print("Saved:",OUTFIG)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp

# ======================================================
# Config
# ======================================================
GSEA_OUTDIR = "fig2_final/GSEA_unbiased"


CHD_GSEA_CSV = os.path.join(GSEA_OUTDIR, "GSEA_CHD_raw.csv")
IST_GSEA_CSV = os.path.join(GSEA_OUTDIR, "GSEA_iStroke_raw.csv")

CHD_RNK = os.path.join(GSEA_OUTDIR, "CHD_GSEA_ranked_list.rnk")
IST_RNK = os.path.join(GSEA_OUTDIR, "iStroke_GSEA_ranked_list.rnk")


GO_SET = "GO_Biological_Process_2021"

# output
PLOT_OUTDIR = os.path.join(GSEA_OUTDIR, "barstyle_plots_from_existing_results")
os.makedirs(PLOT_OUTDIR, exist_ok=True)

# colors
DATASET_COLORS = {
    "CHD": "#838eac",
    "iStroke": "#ea977c",
}

# plot parameters
Q_CUTOFF = 0.05
MAX_SIG_TERMS_PER_FIG = 40
BAR_N_BINS = 90
FIG_WIDTH = 8.0
ROW_HEIGHT = 0.60


# ======================================================
# Utils
# ======================================================
def find_fdr_col(df):
    for c in ["FDR q-val", "FDR", "fdr", "fdr_qval", "qval", "Adjusted P-value"]:
        if c in df.columns:
            return c
    raise ValueError(f"Cannot find FDR column. Columns are: {df.columns.tolist()}")


def normalize_gsea_df(df):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    df = df.copy()

    if "Term" not in df.columns:
        df = df.reset_index()
        if "index" in df.columns and "Term" not in df.columns:
            df = df.rename(columns={"index": "Term"})

    fdr_col = find_fdr_col(df)

    if "NES" not in df.columns:
        raise ValueError(f"Cannot find NES column. Columns are: {df.columns.tolist()}")

    df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
    df[fdr_col] = pd.to_numeric(df[fdr_col], errors="coerce")

    df = df.dropna(subset=["Term", "NES", fdr_col]).copy()
    df = df.rename(columns={fdr_col: "FDR"})

    df["FDR"] = df["FDR"].clip(lower=1e-300)

    return df


def read_rnk(rnk_path):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    rnk = pd.read_csv(
        rnk_path,
        sep="\t",
        header=None,
        names=["gene", "score"]
    )

    rnk["gene"] = rnk["gene"].astype(str)
    rnk["score"] = pd.to_numeric(rnk["score"], errors="coerce")
    rnk = rnk.dropna(subset=["gene", "score"])
    rnk = rnk.drop_duplicates("gene")
    rnk = rnk.sort_values("score", ascending=False)

    return rnk


def load_go_library():
    """Analysis helper. Inputs and outputs follow the function implementation."""

    gene_sets = gp.get_library(
        name=GO_SET,
        organism="Human"
    )

    gene_sets = {
        str(term): set(str(g) for g in genes)
        for term, genes in gene_sets.items()
    }

    print(f"Loaded GO gene sets: {len(gene_sets)}")

    return gene_sets


def make_term_profile_from_rnk(rnk, gene_sets, term, n_bins=90):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    ranking_genes = rnk["gene"].astype(str).tolist()
    n_genes = len(ranking_genes)

    if term not in gene_sets:
        print(f"[Warning] Term not found in GO library: {term}")
        return np.zeros(n_bins)

    geneset = gene_sets[term]

    hits = np.array(
        [i for i, g in enumerate(ranking_genes) if g in geneset],
        dtype=int
    )

    if len(hits) == 0:
        return np.zeros(n_bins)

    bins = np.linspace(0, n_genes, n_bins + 1)
    counts, _ = np.histogram(hits, bins=bins)

    profile = counts.astype(float)

    
    kernel = np.array([1, 2, 3, 2, 1], dtype=float)
    kernel = kernel / kernel.sum()
    profile = np.convolve(profile, kernel, mode="same")

    if profile.max() > 0:
        profile = profile / profile.max()

    return profile


def clean_go_label(term):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    term = str(term)

    
    # term = re.sub(r"\s*\(GO:\d+\)", "", term)

    return term


# ======================================================
# Main plotting function
# ======================================================
def plot_gsea_barstyle_from_existing_result(
    gsea_csv,
    rnk_path,
    dataset_label,
    color,
    outfile,
    q_cutoff=0.05,
    max_terms=40,
):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    gsea_df = pd.read_csv(gsea_csv)
    gsea_df = normalize_gsea_df(gsea_df)

    rnk = read_rnk(rnk_path)
    gene_sets = load_go_library()

    sig = gsea_df[gsea_df["FDR"] < q_cutoff].copy()

    print(f"\n{dataset_label}")
    print(f"Total GSEA terms: {gsea_df.shape[0]}")
    print(f"Significant terms q < {q_cutoff}: {sig.shape[0]}")

    if sig.empty:
        print(f"[Warning] No significant pathways for {dataset_label}. No figure saved.")
        return

    
    sig_out = os.path.join(
        PLOT_OUTDIR,
        f"{dataset_label}_GO_BP_significant_q{q_cutoff}.csv"
    )
    sig.sort_values("FDR").to_csv(sig_out, index=False)
    print(f"[Saved] {sig_out}")

    
    pos = sig[sig["NES"] > 0].sort_values("NES", ascending=False)
    neg = sig[sig["NES"] < 0].sort_values("NES", ascending=True)

    df_plot = pd.concat([pos, neg], axis=0).head(max_terms).copy()

    n_terms = len(df_plot)
    fig_h = max(2.5, ROW_HEIGHT * n_terms + 0.8)

    fig, axes = plt.subplots(
        n_terms,
        1,
        figsize=(FIG_WIDTH, fig_h),
        sharex=True
    )

    if n_terms == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, df_plot.iterrows()):
        term = row["Term"]
        term_label = clean_go_label(term)

        nes = float(row["NES"])
        qval = float(row["FDR"])

        profile = make_term_profile_from_rnk(
            rnk=rnk,
            gene_sets=gene_sets,
            term=term,
            n_bins=BAR_N_BINS
        )

        x = np.arange(BAR_N_BINS)

        ax.bar(
            x,
            profile,
            width=0.85,
            color=color,
            edgecolor=color,
            linewidth=0.15
        )

        ax.axhline(0, color="0.70", linewidth=0.6)

        ax.text(
            0.0,
            1.15,
            term_label,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=9.5,
            fontstyle="italic",
            fontweight="bold" if qval < 0.01 else "normal"
        )

        ax.text(
            0.995,
            1.15,
            f"NES = {nes:.2f}    q = {qval:.2g}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=9.5
        )

        ax.set_ylim(0, 1.25)
        ax.set_yticks([])
        ax.set_xticks([])

        for side in ["top", "right", "left"]:
            ax.spines[side].set_visible(False)

        ax.spines["bottom"].set_linewidth(0.65)
        ax.spines["bottom"].set_color("0.50")

    fig.suptitle(
        f"{dataset_label} – GO BP GSEA significant pathways",
        fontsize=11,
        y=1.005
    )

    plt.tight_layout(h_pad=0.65)

    plt.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

    plt.close()

    print(f"[Saved] {outfile}")


# ======================================================
# Optional: positive / negative separated plots
# ======================================================
def plot_gsea_barstyle_split_from_existing_result(
    gsea_csv,
    rnk_path,
    dataset_label,
    color,
    q_cutoff=0.05,
    max_terms=40,
):
    gsea_df = pd.read_csv(gsea_csv)
    gsea_df = normalize_gsea_df(gsea_df)

    rnk = read_rnk(rnk_path)
    gene_sets = load_go_library()

    sig = gsea_df[gsea_df["FDR"] < q_cutoff].copy()

    if sig.empty:
        return

    for direction, subdf in [
        ("positive_NES", sig[sig["NES"] > 0].sort_values("NES", ascending=False)),
        ("negative_NES", sig[sig["NES"] < 0].sort_values("NES", ascending=True)),
    ]:
        if subdf.empty:
            continue

        tmp_csv = os.path.join(
            PLOT_OUTDIR,
            f"{dataset_label}_GO_BP_{direction}_q{q_cutoff}_tmp.csv"
        )
        subdf.to_csv(tmp_csv, index=False)

        tmp_gsea_csv = tmp_csv

        outfile = os.path.join(
            PLOT_OUTDIR,
            f"fig2_GSEA_{dataset_label}_GO_BP_{direction}_barstyle.pdf"
        )

        plot_gsea_barstyle_from_existing_result(
            gsea_csv=tmp_gsea_csv,
            rnk_path=rnk_path,
            dataset_label=f"{dataset_label} {direction}",
            color=color,
            outfile=outfile,
            q_cutoff=q_cutoff,
            max_terms=max_terms,
        )


# ======================================================
# Run
# ======================================================
if __name__ == "__main__":

    plot_gsea_barstyle_from_existing_result(
        gsea_csv=CHD_GSEA_CSV,
        rnk_path=CHD_RNK,
        dataset_label="CHD",
        color=DATASET_COLORS["CHD"],
        outfile=os.path.join(
            PLOT_OUTDIR,
            "fig2_GSEA_CHD_GO_BP_significant_barstyle.pdf"
        ),
        q_cutoff=Q_CUTOFF,
        max_terms=MAX_SIG_TERMS_PER_FIG,
    )

    plot_gsea_barstyle_from_existing_result(
        gsea_csv=IST_GSEA_CSV,
        rnk_path=IST_RNK,
        dataset_label="iStroke",
        color=DATASET_COLORS["iStroke"],
        outfile=os.path.join(
            PLOT_OUTDIR,
            "fig2_GSEA_iStroke_GO_BP_significant_barstyle.pdf"
        ),
        q_cutoff=Q_CUTOFF,
        max_terms=MAX_SIG_TERMS_PER_FIG,
    )

    
    # plot_gsea_barstyle_split_from_existing_result(
    #     gsea_csv=CHD_GSEA_CSV,
    #     rnk_path=CHD_RNK,
    #     dataset_label="CHD",
    #     color=DATASET_COLORS["CHD"],
    #     q_cutoff=Q_CUTOFF,
    #     max_terms=MAX_SIG_TERMS_PER_FIG,
    # )
    #
    # plot_gsea_barstyle_split_from_existing_result(
    #     gsea_csv=IST_GSEA_CSV,
    #     rnk_path=IST_RNK,
    #     dataset_label="iStroke",
    #     color=DATASET_COLORS["iStroke"],
    #     q_cutoff=Q_CUTOFF,
    #     max_terms=MAX_SIG_TERMS_PER_FIG,
    # )

    print("\nDone.")
    print("Plot output dir:", PLOT_OUTDIR)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp

# ======================================================
# Config
# ======================================================
GSEA_OUTDIR = "fig2_final/GSEA_unbiased"


CHD_GSEA_CSV = os.path.join(GSEA_OUTDIR, "GSEA_CHD_raw.csv")
IST_GSEA_CSV = os.path.join(GSEA_OUTDIR, "GSEA_iStroke_raw.csv")

CHD_RNK = os.path.join(GSEA_OUTDIR, "CHD_GSEA_ranked_list.rnk")
IST_RNK = os.path.join(GSEA_OUTDIR, "iStroke_GSEA_ranked_list.rnk")


GO_SET = "GO_Biological_Process_2021"

# output
PLOT_OUTDIR = os.path.join(GSEA_OUTDIR, "barstyle_plots_from_existing_results")
os.makedirs(PLOT_OUTDIR, exist_ok=True)

# colors
DATASET_COLORS = {
    "CHD": "#838eac",
    "iStroke": "#ea977c",
}

# plot parameters
Q_CUTOFF = 0.05
MAX_SIG_TERMS_PER_FIG = 40
BAR_N_BINS = 90
FIG_WIDTH = 8.0
ROW_HEIGHT = 0.60


# ======================================================
# Utils
# ======================================================
def find_fdr_col(df):
    for c in ["FDR q-val", "FDR", "fdr", "fdr_qval", "qval", "Adjusted P-value"]:
        if c in df.columns:
            return c
    raise ValueError(f"Cannot find FDR column. Columns are: {df.columns.tolist()}")


def normalize_gsea_df(df):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    df = df.copy()

    if "Term" not in df.columns:
        df = df.reset_index()
        if "index" in df.columns and "Term" not in df.columns:
            df = df.rename(columns={"index": "Term"})

    fdr_col = find_fdr_col(df)

    if "NES" not in df.columns:
        raise ValueError(f"Cannot find NES column. Columns are: {df.columns.tolist()}")

    df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
    df[fdr_col] = pd.to_numeric(df[fdr_col], errors="coerce")

    df = df.dropna(subset=["Term", "NES", fdr_col]).copy()
    df = df.rename(columns={fdr_col: "FDR"})

    df["FDR"] = df["FDR"].clip(lower=1e-300)

    return df


def read_rnk(rnk_path):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    rnk = pd.read_csv(
        rnk_path,
        sep="\t",
        header=None,
        names=["gene", "score"]
    )

    rnk["gene"] = rnk["gene"].astype(str)
    rnk["score"] = pd.to_numeric(rnk["score"], errors="coerce")
    rnk = rnk.dropna(subset=["gene", "score"])
    rnk = rnk.drop_duplicates("gene")
    rnk = rnk.sort_values("score", ascending=False)

    return rnk


def load_go_library():
    """Analysis helper. Inputs and outputs follow the function implementation."""

    gene_sets = gp.get_library(
        name=GO_SET,
        organism="Human"
    )

    gene_sets = {
        str(term): set(str(g) for g in genes)
        for term, genes in gene_sets.items()
    }

    print(f"Loaded GO gene sets: {len(gene_sets)}")

    return gene_sets


def make_term_profile_from_rnk(rnk, gene_sets, term, n_bins=90):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    ranking_genes = rnk["gene"].astype(str).tolist()
    n_genes = len(ranking_genes)

    if term not in gene_sets:
        print(f"[Warning] Term not found in GO library: {term}")
        return np.zeros(n_bins)

    geneset = gene_sets[term]

    hits = np.array(
        [i for i, g in enumerate(ranking_genes) if g in geneset],
        dtype=int
    )

    if len(hits) == 0:
        return np.zeros(n_bins)

    bins = np.linspace(0, n_genes, n_bins + 1)
    counts, _ = np.histogram(hits, bins=bins)

    profile = counts.astype(float)

    
    kernel = np.array([1, 2, 3, 2, 1], dtype=float)
    kernel = kernel / kernel.sum()
    profile = np.convolve(profile, kernel, mode="same")

    if profile.max() > 0:
        profile = profile / profile.max()

    return profile


def clean_go_label(term):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    term = str(term)

    
    # term = re.sub(r"\s*\(GO:\d+\)", "", term)

    return term


# ======================================================
# Main plotting function
# ======================================================
def plot_gsea_barstyle_from_existing_result(
    gsea_csv,
    rnk_path,
    dataset_label,
    color,
    outfile,
    q_cutoff=0.05,
    max_terms=40,
):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    gsea_df = pd.read_csv(gsea_csv)
    gsea_df = normalize_gsea_df(gsea_df)

    rnk = read_rnk(rnk_path)
    gene_sets = load_go_library()

    sig = gsea_df[gsea_df["FDR"] < q_cutoff].copy()

    print(f"\n{dataset_label}")
    print(f"Total GSEA terms: {gsea_df.shape[0]}")
    print(f"Significant terms q < {q_cutoff}: {sig.shape[0]}")

    if sig.empty:
        print(f"[Warning] No significant pathways for {dataset_label}. No figure saved.")
        return

    
    sig_out = os.path.join(
        PLOT_OUTDIR,
        f"{dataset_label}_GO_BP_significant_q{q_cutoff}.csv"
    )
    sig.sort_values("FDR").to_csv(sig_out, index=False)
    print(f"[Saved] {sig_out}")

    
    pos = sig[sig["NES"] > 0].sort_values("NES", ascending=False)
    neg = sig[sig["NES"] < 0].sort_values("NES", ascending=True)

    df_plot = pd.concat([pos, neg], axis=0).head(max_terms).copy()

    n_terms = len(df_plot)
    fig_h = max(2.5, ROW_HEIGHT * n_terms + 0.8)

    fig, axes = plt.subplots(
        n_terms,
        1,
        figsize=(FIG_WIDTH, fig_h),
        sharex=True
    )

    if n_terms == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, df_plot.iterrows()):
        term = row["Term"]
        term_label = clean_go_label(term)

        nes = float(row["NES"])
        qval = float(row["FDR"])

        profile = make_term_profile_from_rnk(
            rnk=rnk,
            gene_sets=gene_sets,
            term=term,
            n_bins=BAR_N_BINS
        )

        x = np.arange(BAR_N_BINS)

        ax.bar(
            x,
            profile,
            width=0.85,
            color=color,
            edgecolor=color,
            linewidth=0.15
        )

        ax.axhline(0, color="0.70", linewidth=0.6)

        ax.text(
            0.0,
            1.15,
            term_label,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=9.5,
            fontstyle="italic",
            fontweight="bold" if qval < 0.01 else "normal"
        )

        ax.text(
            0.995,
            1.15,
            f"NES = {nes:.2f}    q = {qval:.2g}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=9.5
        )

        ax.set_ylim(0, 1.25)
        ax.set_yticks([])
        ax.set_xticks([])

        for side in ["top", "right", "left"]:
            ax.spines[side].set_visible(False)

        ax.spines["bottom"].set_linewidth(0.65)
        ax.spines["bottom"].set_color("0.50")

    fig.suptitle(
        f"{dataset_label} – GO BP GSEA significant pathways",
        fontsize=11,
        y=1.005
    )

    plt.tight_layout(h_pad=0.65)

    plt.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

    plt.close()

    print(f"[Saved] {outfile}")


# ======================================================
# Optional: positive / negative separated plots
# ======================================================
def plot_gsea_barstyle_split_from_existing_result(
    gsea_csv,
    rnk_path,
    dataset_label,
    color,
    q_cutoff=0.05,
    max_terms=40,
):
    gsea_df = pd.read_csv(gsea_csv)
    gsea_df = normalize_gsea_df(gsea_df)

    rnk = read_rnk(rnk_path)
    gene_sets = load_go_library()

    sig = gsea_df[gsea_df["FDR"] < q_cutoff].copy()

    if sig.empty:
        return

    for direction, subdf in [
        ("positive_NES", sig[sig["NES"] > 0].sort_values("NES", ascending=False)),
        ("negative_NES", sig[sig["NES"] < 0].sort_values("NES", ascending=True)),
    ]:
        if subdf.empty:
            continue

        tmp_csv = os.path.join(
            PLOT_OUTDIR,
            f"{dataset_label}_GO_BP_{direction}_q{q_cutoff}_tmp.csv"
        )
        subdf.to_csv(tmp_csv, index=False)

        tmp_gsea_csv = tmp_csv

        outfile = os.path.join(
            PLOT_OUTDIR,
            f"fig2_GSEA_{dataset_label}_GO_BP_{direction}_barstyle.pdf"
        )

        plot_gsea_barstyle_from_existing_result(
            gsea_csv=tmp_gsea_csv,
            rnk_path=rnk_path,
            dataset_label=f"{dataset_label} {direction}",
            color=color,
            outfile=outfile,
            q_cutoff=q_cutoff,
            max_terms=max_terms,
        )


# ======================================================
# Run
# ======================================================
if __name__ == "__main__":

    plot_gsea_barstyle_from_existing_result(
        gsea_csv=CHD_GSEA_CSV,
        rnk_path=CHD_RNK,
        dataset_label="CHD",
        color=DATASET_COLORS["CHD"],
        outfile=os.path.join(
            PLOT_OUTDIR,
            "fig2_GSEA_CHD_GO_BP_significant_barstyle.pdf"
        ),
        q_cutoff=Q_CUTOFF,
        max_terms=MAX_SIG_TERMS_PER_FIG,
    )

    plot_gsea_barstyle_from_existing_result(
        gsea_csv=IST_GSEA_CSV,
        rnk_path=IST_RNK,
        dataset_label="iStroke",
        color=DATASET_COLORS["iStroke"],
        outfile=os.path.join(
            PLOT_OUTDIR,
            "fig2_GSEA_iStroke_GO_BP_significant_barstyle.pdf"
        ),
        q_cutoff=Q_CUTOFF,
        max_terms=MAX_SIG_TERMS_PER_FIG,
    )

    
    # plot_gsea_barstyle_split_from_existing_result(
    #     gsea_csv=CHD_GSEA_CSV,
    #     rnk_path=CHD_RNK,
    #     dataset_label="CHD",
    #     color=DATASET_COLORS["CHD"],
    #     q_cutoff=Q_CUTOFF,
    #     max_terms=MAX_SIG_TERMS_PER_FIG,
    # )
    #
    # plot_gsea_barstyle_split_from_existing_result(
    #     gsea_csv=IST_GSEA_CSV,
    #     rnk_path=IST_RNK,
    #     dataset_label="iStroke",
    #     color=DATASET_COLORS["iStroke"],
    #     q_cutoff=Q_CUTOFF,
    #     max_terms=MAX_SIG_TERMS_PER_FIG,
    # )

    print("\nDone.")
    print("Plot output dir:", PLOT_OUTDIR)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from scipy.stats import fisher_exact

# ======================================================
# Matplotlib settings
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ======================================================
# Config
# ======================================================

ENDPOINTS = ["CHD", "istroke"]
HIGH_RISK_Q = 0.85

IG_TEMPLATE   = "code_02_02_save/snp_20260202_1432/explanations/{}_snp_TR_best_IG.csv"
PRED_TEMPLATE = "code_02_02_save/snp_20260202_1432/preds/{}_snp_TR_test_best_pred.csv"

VARIANT_CSV = "data/private/CHD_SNP_variant_gene_with_symbol.csv"
GWAS_CATALOG = "data/private/gwas-catalog-download-associations-v1.0-full.tsv"

OUTDIR = "fig2_final/chromosome_IG_combined_manual_studies_all_hits_final"
os.makedirs(OUTDIR, exist_ok=True)

CYTOBAND_FILE = os.path.join(OUTDIR, "hg38_cytoBand.txt.gz")

OUTFIG = os.path.join(
    OUTDIR,
    "CHD_istroke_selected_chromosomes_label_regions_q75_heatmap.pdf"
)

OUT_COLORBAR = os.path.join(
    OUTDIR,
    "CHD_istroke_selected_chromosomes_label_regions_q75_heatmap_colorbar.pdf"
)

OUT_BAND_TABLE = os.path.join(
    OUTDIR,
    "CHD_istroke_cytoband_summary_with_OR_manual_studies_all_hits_final.csv"
)

OUT_SNP_TABLE = os.path.join(
    OUTDIR,
    "CHD_istroke_snp_band_ig_summary_manual_studies_all_hits_final.csv"
)

OUT_Q75_TABLE = os.path.join(
    OUTDIR,
    "CHD_istroke_q75_selected_bands_manual_studies_all_hits_final.csv"
)

OUT_SELECTED_STUDIES = os.path.join(
    OUTDIR,
    "manual_selected_CHD_istroke_GWAS_hits_all_hits_final.csv"
)

OUT_SELECTED_STUDY_SUMMARY = os.path.join(
    OUTDIR,
    "manual_selected_CHD_istroke_GWAS_study_summary_all_hits_final.csv"
)

CHR_ORDER = [f"chr{i}" for i in range(1, 23)] + ["chrX"]

# ======================================================


# ======================================================

MANUAL_SELECTED_STUDIES = [
    # istroke
    {"endpoint": "istroke", "FIRST AUTHOR": "Malik R",      "DATE": "2018-03-12", "JOURNAL": "Nat Genet", "PUBMEDID": "29531354"},
    {"endpoint": "istroke", "FIRST AUTHOR": "Mishra A",     "DATE": "2022-09-30", "JOURNAL": "Nature",    "PUBMEDID": "36180795"},
    {"endpoint": "istroke", "FIRST AUTHOR": "Soderholm M",  "DATE": "2019-02-22", "JOURNAL": "Neurology", "PUBMEDID": "30796134"},

    # CHD
    {"endpoint": "CHD", "FIRST AUTHOR": "Aragam KG",      "DATE": "2022-12-06", "JOURNAL": "Nat Genet", "PUBMEDID": "36474045"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Tcheandjieu C",  "DATE": "2022-08-01", "JOURNAL": "Nat Med",   "PUBMEDID": "35915156"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Koyama S",       "DATE": "2020-10-05", "JOURNAL": "Nat Genet", "PUBMEDID": "33020668"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Lu X",           "DATE": "2012-07-01", "JOURNAL": "Nat Genet", "PUBMEDID": "22751097"},
    {"endpoint": "CHD", "FIRST AUTHOR": "Nikpay M",       "DATE": "2015-09-07", "JOURNAL": "Nat Genet", "PUBMEDID": "26343387"},
]

# ======================================================



# ======================================================

LABEL_REGIONS = [
    ("chr9",  "p21",    "p21",    "9p21 / CDKN2A-CDKN2B-ANRIL"),
    ("chr7",  "p21",    "p21",    "7p21 / HDAC9"),
    ("chr6",  "p24.1",  "p24.1",  "6p24.1 / PHACTR1"),
    ("chr6",  "q26",  "q26",  "6q26 / LPA"),

    # LDLR is at 19p13.2, so label should be 19p13, not 19q13
    ("chr19", "p13",    "p13",    "19p13 / LDLR"),

    ("chr1",  "p13",    "p13",    "1p13 / SORT1"),
    ("chr1",  "p32",    "p32",    "1p32 / PCSK9"),

    ("chr9",  "q33.2",  "q33.2",  "9q33.2"),

    ("chr5",  "q22",    "q22",    "5q22"),
    ("chr10", "q23.31", "q23.31", "10q23.31 / LIPA"),
    ("chr16", "q24.2",  "q24.2",  "16q24.2"),
]

PLOT_CHROMS_FROM_LABELS_ONLY = True

# ======================================================
# Plot config
# ======================================================

FIGSIZE = (16, 8)

CHROM_HEIGHT = 0.22
TRACK_HEIGHT = 0.10
TRACK_GAP = 0.035
CHROM_TO_TRACK_GAP = 0.06
Y_GAP = 0.82

#HEATMAP_CMAP = "viridis"
HEATMAP_CMAP = LinearSegmentedColormap.from_list(
    "ig_grey_black",
    ["#EEEEEE", "#111111"]
)

#COLOR_NONE = "#E9E9E9"
COLOR_NONE = "#FFFFFF"
COLOR_CHD = "#D55E5E"
COLOR_ISTROKE = "#5B8FD9"
COLOR_BOTH = "#9B59B6"
EDGE_COLOR = "#A8A8A8"

LABEL_EDGE_COLOR = "#D62728"
LABEL_TEXT_COLOR = "#B22222"
LABEL_LINEWIDTH = 1.4
LABEL_FONTSIZE = 8

PLOT_IG_Q = 0.75


HIGHLIGHT_MODE = "any_hit"
MIN_OR = 1.0
MAX_P = 0.05
MIN_HITS = 1

# ======================================================

# ======================================================

CHD_PATTERNS = [
    r"\bcoronary heart disease\b",
    r"\bcoronary artery disease\b",
    r"\bischemic heart disease\b",
    r"\bmyocardial infarction\b",
    r"\bangina\b",
    r"\bunstable angina\b",
    r"\bacute coronary syndrome\b",
    r"\batherosclerosis\b",
    r"\bcoronary atherosclerosis\b",
    r"\bcoronary stenosis\b",
    r"\bchd\b",
    r"\bcad\b",
]

ISTROKE_PATTERNS = [
    r"\bischemic stroke\b",
    r"\bischaemic stroke\b",
    r"\bstroke\b",
    r"\bcerebral infarction\b",
    r"\bbrain infarction\b",
    r"\blacunar stroke\b",
    r"\bcardioembolic stroke\b",
    r"\blarge artery stroke\b",
    r"\bsmall vessel stroke\b",
    r"\bsmall artery occlusion\b",
    r"\btransient ischemic attack\b",
    r"\btransient ischaemic attack\b",
    r"\btia\b",
]

# ======================================================
# Helpers
# ======================================================

def download_cytoband_if_needed(path_out):
    if os.path.exists(path_out):
        return
    url = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cytoBand.txt.gz"
    print("Downloading cytoband file from:", url)
    urllib.request.urlretrieve(url, path_out)
    print("Saved:", path_out)


def parse_variant(v):
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    return np.nan, np.nan


def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_date_to_yyyy_mm_dd(x):
    dt = pd.to_datetime(x, errors="coerce")
    if pd.isna(dt):
        return normalize_text(x)
    return dt.strftime("%Y-%m-%d")


def band_prefix(band_name):
    """
    p21.1 / p21.2 / p21.3 -> p21
    q22.1 / q22.2 / q22.3 -> q22
    q13.11 -> q13
    """
    m = re.match(r"^([pq]\d+)", str(band_name))
    if m:
        return m.group(1)
    return str(band_name)


def is_coarse_band_name(x):
    """
    True for p21/q22/q13, False for p21.1/q22.3/q13.11
    """
    return re.match(r"^[pq]\d+$", str(x)) is not None


def build_regex(patterns):
    return re.compile("|".join(patterns), flags=re.IGNORECASE)


def trait_match(series, regex_obj):
    return series.fillna("").astype(str).str.contains(regex_obj, regex=True, na=False)


def load_variant_positions(variant_csv):
    variant = pd.read_csv(variant_csv)
    variant[["chr", "pos"]] = variant["variant_id"].apply(
        lambda x: pd.Series(parse_variant(x))
    )
    variant = variant.dropna(subset=["chr", "pos"]).copy()
    variant["pos"] = variant["pos"].astype(int)
    variant = variant[variant["chr"].isin(CHR_ORDER)].copy()
    return variant


def load_endpoint_ig(endpoint, variant_df, high_risk_q=0.85):
    ig_csv = IG_TEMPLATE.format(endpoint)
    pred_csv = PRED_TEMPLATE.format(endpoint)

    ig = pd.read_csv(ig_csv)

    if "ig_abs" in ig.columns:
        ig["abs_ig"] = ig["ig_abs"].abs()
    elif "ig" in ig.columns:
        ig["abs_ig"] = ig["ig"].abs()
    else:
        raise ValueError(f"{ig_csv} must contain 'ig_abs' or 'ig'.")

    if high_risk_q is not None:
        pred = pd.read_csv(pred_csv)
        pred["pid"] = pred["ID"]
        risk_cut = pred["risk"].quantile(high_risk_q)
        high_pid = pred.loc[pred["risk"] >= risk_cut, "pid"]
        ig = ig[ig["pid"].isin(high_pid)].copy()

    ig = ig.merge(
        variant_df[["SNP", "chr", "pos"]],
        left_on="feature",
        right_on="SNP",
        how="inner"
    )

    snp_agg = (
        ig.groupby(["SNP", "chr", "pos"], as_index=False)["abs_ig"]
          .mean()
          .rename(columns={"abs_ig": "mean_abs_ig"})
    )
    return snp_agg


def load_cytobands(path_in):
    cyto = pd.read_csv(
        path_in,
        sep="\t",
        compression="gzip",
        header=None,
        names=["chrom", "start", "end", "band_name", "gieStain"]
    )
    cyto = cyto[cyto["chrom"].isin(CHR_ORDER)].copy()
    cyto = cyto.sort_values(["chrom", "start"]).reset_index(drop=True)

    cyto["band_id"] = (
        cyto["chrom"].astype(str) + ":" +
        cyto["band_name"].astype(str) + ":" +
        cyto["start"].astype(str) + "-" +
        cyto["end"].astype(str)
    )
    cyto["start_mb"] = cyto["start"] / 1e6
    cyto["end_mb"] = cyto["end"] / 1e6
    cyto["len_bp"] = cyto["end"] - cyto["start"]
    cyto["len_mb"] = cyto["len_bp"] / 1e6
    cyto["band_prefix"] = cyto["band_name"].apply(band_prefix)
    return cyto


def map_positions_to_bands(df, bands_df, chrom_col="chr", pos_col="pos"):
    if df.empty:
        return df.copy()

    out = []

    for chrom, sub in df.groupby(chrom_col):
        band_sub = bands_df[bands_df["chrom"] == chrom].sort_values("start").copy()

        if band_sub.empty:
            tmp = sub.copy()
            tmp["band_id"] = np.nan
            tmp["band_name"] = np.nan
            out.append(tmp)
            continue

        starts = band_sub["start"].to_numpy()
        ends = band_sub["end"].to_numpy()
        band_ids = band_sub["band_id"].to_numpy()
        band_names = band_sub["band_name"].to_numpy()

        pos = sub[pos_col].to_numpy()
        idx = np.searchsorted(starts, pos, side="right") - 1

        band_id_arr = np.array([np.nan] * len(sub), dtype=object)
        band_name_arr = np.array([np.nan] * len(sub), dtype=object)

        valid = idx >= 0
        valid_idx = idx[valid]
        valid_pos = pos[valid]
        inside = valid_pos < ends[valid_idx]

        pos_valid_indices = np.where(valid)[0][inside]
        band_valid_indices = valid_idx[inside]

        band_id_arr[pos_valid_indices] = band_ids[band_valid_indices]
        band_name_arr[pos_valid_indices] = band_names[band_valid_indices]

        tmp = sub.copy()
        tmp["band_id"] = band_id_arr
        tmp["band_name"] = band_name_arr
        out.append(tmp)

    out = pd.concat(out, ignore_index=True)
    return out


def compute_band_enrichment(band_table, disease_col_hits, total_col_hits, prefix):
    disease_total = band_table[disease_col_hits].sum()
    all_total = band_table[total_col_hits].sum()
    other_total = all_total - disease_total

    odds_list = []
    pval_list = []

    for _, row in band_table.iterrows():
        a = int(row[disease_col_hits])
        total_here = int(row[total_col_hits])
        b = max(total_here - a, 0)
        c = max(disease_total - a, 0)
        d = max(other_total - b, 0)

        table = np.array([[a, b], [c, d]], dtype=int)

        try:
            odds_ratio, p_value = fisher_exact(table, alternative="greater")
        except Exception:
            odds_ratio, p_value = np.nan, np.nan

        odds_list.append(odds_ratio)
        pval_list.append(p_value)

    band_table[f"{prefix}_odds_ratio"] = odds_list
    band_table[f"{prefix}_pvalue"] = pval_list
    return band_table


def decide_highlight(hit_count, or_value, p_value):
    if HIGHLIGHT_MODE == "any_hit":
        return hit_count >= 1

    if HIGHLIGHT_MODE == "or_and_p":
        if pd.isna(or_value) or pd.isna(p_value):
            return False
        return (hit_count >= MIN_HITS) and (or_value >= MIN_OR) and (p_value < MAX_P)

    return hit_count >= 1


def band_region_type(row):
    if row["chd_highlight"] and row["istroke_highlight"]:
        return "both"
    if row["chd_highlight"]:
        return "chd"
    if row["istroke_highlight"]:
        return "istroke"
    return "none"


def chrom_sort_key(chrom):
    x = str(chrom).replace("chr", "")
    if x == "X":
        return 23
    if x == "Y":
        return 24
    return int(x)


def get_region_span(bands_df, chrom, start_band, end_band):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    sub_chr = bands_df[bands_df["chrom"] == chrom].sort_values("start").copy()

    if sub_chr.empty:
        print(f"[Warning] Cannot find chromosome: {chrom}")
        return None

    # coarse mode: p21, q22, q13...
    if is_coarse_band_name(start_band) and start_band == end_band:
        region_bands = sub_chr[sub_chr["band_prefix"] == start_band].copy()

        if region_bands.empty:
            print(f"[Warning] Cannot find coarse region: {chrom} {start_band}")
            return None

        return {
            "chrom": chrom,
            "start_band": start_band,
            "end_band": end_band,
            "start_mb": region_bands["start_mb"].min(),
            "end_mb": region_bands["end_mb"].max(),
            "included_bands": ",".join(region_bands["band_name"].tolist()),
        }

    # exact single / exact range mode
    start_row = sub_chr[sub_chr["band_name"] == start_band]
    end_row = sub_chr[sub_chr["band_name"] == end_band]

    if start_row.empty or end_row.empty:
        print(f"[Warning] Cannot find region: {chrom} {start_band}-{end_band}")
        return None

    left_bp = min(int(start_row["start"].iloc[0]), int(end_row["start"].iloc[0]))
    right_bp = max(int(start_row["end"].iloc[0]), int(end_row["end"].iloc[0]))

    region_bands = sub_chr[
        (sub_chr["end"] > left_bp) &
        (sub_chr["start"] < right_bp)
    ].copy()

    if region_bands.empty:
        return None

    return {
        "chrom": chrom,
        "start_band": start_band,
        "end_band": end_band,
        "start_mb": region_bands["start_mb"].min(),
        "end_mb": region_bands["end_mb"].max(),
        "included_bands": ",".join(region_bands["band_name"].tolist()),
    }


def apply_manual_study_flags_all_hits(gwas_band, manual_studies):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    gwas_band = gwas_band.copy()

    gwas_band["is_chd_related"] = False
    gwas_band["is_istroke_related"] = False
    gwas_band["manual_study_selected"] = False

    for col in ["FIRST AUTHOR", "JOURNAL", "PUBMEDID"]:
        if col in gwas_band.columns:
            gwas_band[f"__{col}"] = gwas_band[col].apply(normalize_text)
        else:
            gwas_band[f"__{col}"] = ""

    gwas_band["__DATE"] = gwas_band["DATE"].apply(normalize_date_to_yyyy_mm_dd)

    trait_series = gwas_band["DISEASE/TRAIT"].fillna("").astype(str)
    study_series = gwas_band.get(
        "STUDY",
        pd.Series([""] * len(gwas_band), index=gwas_band.index)
    ).fillna("").astype(str)

    chd_regex = build_regex(CHD_PATTERNS)
    istroke_regex = build_regex(ISTROKE_PATTERNS)

    chd_trait_mask = (
        trait_match(trait_series, chd_regex) |
        trait_match(study_series, chd_regex)
    )

    istroke_trait_mask = (
        trait_match(trait_series, istroke_regex) |
        trait_match(study_series, istroke_regex)
    )

    for item in manual_studies:
        endpoint = item["endpoint"]
        author = normalize_text(item["FIRST AUTHOR"])
        date = normalize_date_to_yyyy_mm_dd(item["DATE"])
        journal = normalize_text(item["JOURNAL"])
        pubmedid = normalize_text(item["PUBMEDID"])

        study_mask = (
            (gwas_band["__FIRST AUTHOR"] == author) &
            (gwas_band["__DATE"] == date) &
            (gwas_band["__JOURNAL"] == journal) &
            (gwas_band["__PUBMEDID"] == pubmedid)
        )

        if endpoint == "CHD":
            final_mask = study_mask & chd_trait_mask
            gwas_band.loc[final_mask, "is_chd_related"] = True

        elif endpoint == "istroke":
            final_mask = study_mask & istroke_trait_mask
            gwas_band.loc[final_mask, "is_istroke_related"] = True

        else:
            raise ValueError(f"Unknown endpoint in MANUAL_SELECTED_STUDIES: {endpoint}")

        gwas_band.loc[study_mask, "manual_study_selected"] = True

        print(
            f"[Manual study] {endpoint} | {author} {date} {journal} PMID={pubmedid} | "
            f"all hits in study = {int(study_mask.sum())} | "
            f"endpoint-matched hits = {int(final_mask.sum())}"
        )

    helper_cols = [c for c in gwas_band.columns if c.startswith("__")]
    gwas_band = gwas_band.drop(columns=helper_cols)
    return gwas_band


# ======================================================
# Main calculation
# ======================================================

download_cytoband_if_needed(CYTOBAND_FILE)

bands = load_cytobands(CYTOBAND_FILE)
variant = load_variant_positions(VARIANT_CSV)

chrom_sizes = (
    bands.groupby("chrom", as_index=False)["end_mb"]
         .max()
         .rename(columns={"end_mb": "chrom_len_mb"})
)
chrom_len_dict = dict(zip(chrom_sizes["chrom"], chrom_sizes["chrom_len_mb"]))

# ---------- load CHD / istroke IG ----------
snp_chd = load_endpoint_ig("CHD", variant, high_risk_q=HIGH_RISK_Q)
snp_istroke = load_endpoint_ig("istroke", variant, high_risk_q=HIGH_RISK_Q)

# ---------- map SNP IG to cytobands ----------
snp_chd_band = map_positions_to_bands(snp_chd, bands, chrom_col="chr", pos_col="pos")
snp_istroke_band = map_positions_to_bands(snp_istroke, bands, chrom_col="chr", pos_col="pos")

# ---------- load GWAS catalog ----------
gwas = pd.read_csv(GWAS_CATALOG, sep="\t", low_memory=False)
gwas.columns = gwas.columns.str.strip()
gwas["CHR_ID"] = pd.to_numeric(gwas["CHR_ID"], errors="coerce")
gwas["CHR_POS"] = pd.to_numeric(gwas["CHR_POS"], errors="coerce")
gwas = gwas.dropna(subset=["CHR_ID", "CHR_POS"]).copy()
gwas["CHR_ID"] = gwas["CHR_ID"].astype(int)
gwas["chr"] = "chr" + gwas["CHR_ID"].astype(str)
gwas = gwas[gwas["chr"].isin(CHR_ORDER)].copy()
gwas = gwas.rename(columns={"CHR_POS": "pos"})

# ---------- map GWAS hits to cytobands ----------
gwas_band = map_positions_to_bands(gwas, bands, chrom_col="chr", pos_col="pos")
gwas_band = gwas_band.dropna(subset=["band_id"]).copy()

# ---------- select manual studies, all CHD / istroke hits within these studies ----------
gwas_band = apply_manual_study_flags_all_hits(
    gwas_band=gwas_band,
    manual_studies=MANUAL_SELECTED_STUDIES
)

manual_selected_hits = gwas_band[
    gwas_band["is_chd_related"] | gwas_band["is_istroke_related"]
].copy()
manual_selected_hits.to_csv(OUT_SELECTED_STUDIES, index=False)

print("\nSaved manual selected GWAS hits:")
print(OUT_SELECTED_STUDIES)

if not manual_selected_hits.empty:
    summary_group_cols = [
        c for c in [
            "is_chd_related",
            "is_istroke_related",
            "DISEASE/TRAIT",
            "STUDY",
            "FIRST AUTHOR",
            "DATE",
            "JOURNAL",
            "PUBMEDID",
        ] if c in manual_selected_hits.columns
    ]

    manual_study_summary = (
        manual_selected_hits
        .groupby(summary_group_cols, dropna=False)
        .agg(
            n_gwas_hits=("SNPS", "count") if "SNPS" in manual_selected_hits.columns else ("DISEASE/TRAIT", "count"),
            min_pvalue=("P-VALUE", "min") if "P-VALUE" in manual_selected_hits.columns else ("DISEASE/TRAIT", "count")
        )
        .reset_index()
    )
    manual_study_summary.to_csv(OUT_SELECTED_STUDY_SUMMARY, index=False)
    print("\nSaved manual selected study summary:")
    print(OUT_SELECTED_STUDY_SUMMARY)

# ---------- summarize per band ----------
band_table = bands[
    ["band_id", "chrom", "band_name", "start", "end", "start_mb", "end_mb", "len_bp", "len_mb"]
].copy()

all_counts = (
    gwas_band.groupby("band_id", as_index=False).size()
             .rename(columns={"size": "all_gwas_hits"})
)

chd_counts = (
    gwas_band[gwas_band["is_chd_related"]]
    .groupby("band_id", as_index=False).size()
    .rename(columns={"size": "chd_gwas_hits"})
)

istroke_counts = (
    gwas_band[gwas_band["is_istroke_related"]]
    .groupby("band_id", as_index=False).size()
    .rename(columns={"size": "istroke_gwas_hits"})
)

band_table = band_table.merge(all_counts, on="band_id", how="left")
band_table = band_table.merge(chd_counts, on="band_id", how="left")
band_table = band_table.merge(istroke_counts, on="band_id", how="left")

for col in ["all_gwas_hits", "chd_gwas_hits", "istroke_gwas_hits"]:
    band_table[col] = band_table[col].fillna(0).astype(int)

# ---------- compute OR ----------
band_table = compute_band_enrichment(
    band_table=band_table,
    disease_col_hits="chd_gwas_hits",
    total_col_hits="all_gwas_hits",
    prefix="chd"
)
band_table = compute_band_enrichment(
    band_table=band_table,
    disease_col_hits="istroke_gwas_hits",
    total_col_hits="all_gwas_hits",
    prefix="istroke"
)

# ---------- band-level IG summaries ----------
chd_ig_band = (
    snp_chd_band.dropna(subset=["band_id"])
    .groupby("band_id", as_index=False)["mean_abs_ig"]
    .mean()
    .rename(columns={"mean_abs_ig": "chd_mean_abs_ig"})
)

istroke_ig_band = (
    snp_istroke_band.dropna(subset=["band_id"])
    .groupby("band_id", as_index=False)["mean_abs_ig"]
    .mean()
    .rename(columns={"mean_abs_ig": "istroke_mean_abs_ig"})
)

band_table = band_table.merge(chd_ig_band, on="band_id", how="left")
band_table = band_table.merge(istroke_ig_band, on="band_id", how="left")

# ---------- decide region type ----------
band_table["chd_highlight"] = band_table.apply(
    lambda r: decide_highlight(
        hit_count=r["chd_gwas_hits"],
        or_value=r["chd_odds_ratio"],
        p_value=r["chd_pvalue"]
    ),
    axis=1
)

band_table["istroke_highlight"] = band_table.apply(
    lambda r: decide_highlight(
        hit_count=r["istroke_gwas_hits"],
        or_value=r["istroke_odds_ratio"],
        p_value=r["istroke_pvalue"]
    ),
    axis=1
)

band_table["region_type"] = band_table.apply(band_region_type, axis=1)

# ---------- save summary tables ----------
band_table = band_table.sort_values(["chrom", "start"]).reset_index(drop=True)
band_table.to_csv(OUT_BAND_TABLE, index=False)

snp_band_summary = (
    snp_chd_band[["SNP", "chr", "pos", "mean_abs_ig", "band_id"]]
    .rename(columns={"mean_abs_ig": "chd_mean_abs_ig"})
    .merge(
        snp_istroke_band[["SNP", "chr", "pos", "mean_abs_ig", "band_id"]]
        .rename(columns={"mean_abs_ig": "istroke_mean_abs_ig"}),
        on=["SNP", "chr", "pos", "band_id"],
        how="outer"
    )
)
snp_band_summary.to_csv(OUT_SNP_TABLE, index=False)

print("Saved band summary with OR:", OUT_BAND_TABLE)
print("Saved SNP-band IG table:", OUT_SNP_TABLE)

# ======================================================
# Plot-only selection
# ======================================================

chd_q75 = band_table["chd_mean_abs_ig"].dropna().quantile(PLOT_IG_Q)
istroke_q75 = band_table["istroke_mean_abs_ig"].dropna().quantile(PLOT_IG_Q)

band_table["chd_q75_region"] = (
    band_table["chd_mean_abs_ig"].notna() &
    (band_table["chd_mean_abs_ig"] >= chd_q75)
)

band_table["istroke_q75_region"] = (
    band_table["istroke_mean_abs_ig"].notna() &
    (band_table["istroke_mean_abs_ig"] >= istroke_q75)
)

band_table["any_q75_region"] = (
    band_table["chd_q75_region"] | band_table["istroke_q75_region"]
)

q75_table = band_table[band_table["any_q75_region"]].copy()
q75_table.to_csv(OUT_Q75_TABLE, index=False)


label_chroms = [x[0] for x in LABEL_REGIONS]

if PLOT_CHROMS_FROM_LABELS_ONLY:
    plot_chroms = sorted(set(label_chroms), key=chrom_sort_key)
else:
    plot_chroms = q75_table["chrom"].drop_duplicates().tolist()
    plot_chroms = sorted(set(plot_chroms) | set(label_chroms), key=chrom_sort_key)

print("\nCHD q75 cutoff:", chd_q75)
print("istroke q75 cutoff:", istroke_q75)
print("Chromosomes kept for plotting:", plot_chroms)
print("Saved q75 selected bands:", OUT_Q75_TABLE)


ig_values_all = pd.concat(
    [band_table["chd_mean_abs_ig"], band_table["istroke_mean_abs_ig"]],
    axis=0
).dropna()

vmin = ig_values_all.quantile(0.01)
vmax = ig_values_all.quantile(0.99)
if vmin == vmax:
    vmin = ig_values_all.min()
    vmax = ig_values_all.max()

norm = Normalize(vmin=vmin, vmax=vmax)
#cmap = plt.get_cmap(HEATMAP_CMAP)
cmap = HEATMAP_CMAP

label_region_spans = []
for chrom, start_band, end_band, label in LABEL_REGIONS:
    span = get_region_span(bands, chrom, start_band, end_band)
    if span is not None:
        span["label"] = label
        label_region_spans.append(span)

# ======================================================
# Main figure WITHOUT colorbar
# ======================================================

fig_height = max(4.5, 0.75 * len(plot_chroms) + 2.0)
fig, ax = plt.subplots(figsize=(FIGSIZE[0], fig_height))

y_positions = {chrom: i * Y_GAP for i, chrom in enumerate(plot_chroms)}
max_chr_len = max([chrom_len_dict[c] for c in plot_chroms])

label_offset_counter = {chrom: 0 for chrom in plot_chroms}

for chrom in plot_chroms:

    if chrom not in chrom_len_dict:
        continue

    y0 = y_positions[chrom]
    sub = band_table[band_table["chrom"] == chrom].sort_values("start").copy()

    if sub.empty:
        continue

    
    for _, row in sub.iterrows():
        x = row["start_mb"]
        w = row["len_mb"]

        if row["region_type"] == "chd":
            face = COLOR_CHD
        elif row["region_type"] == "istroke":
            face = COLOR_ISTROKE
        elif row["region_type"] == "both":
            face = COLOR_BOTH
        else:
            face = COLOR_NONE

        ax.add_patch(
            Rectangle(
                (x, y0 - CHROM_HEIGHT / 2),
                w,
                CHROM_HEIGHT,
                facecolor=face,
                edgecolor="white",
                linewidth=0.15,
                zorder=2
            )
        )

    
    ax.add_patch(
        Rectangle(
            (0, y0 - CHROM_HEIGHT / 2),
            chrom_len_dict[chrom],
            CHROM_HEIGHT,
            facecolor="none",
            edgecolor=EDGE_COLOR,
            linewidth=0.8,
            zorder=3
        )
    )

    # heatmap tracks
    chd_track_y = y0 + CHROM_HEIGHT / 2 + CHROM_TO_TRACK_GAP
    istroke_track_y = chd_track_y + TRACK_HEIGHT + TRACK_GAP

    for _, row in sub.iterrows():
        x = row["start_mb"]
        w = row["len_mb"]

        
        if row["chd_q75_region"] and pd.notna(row["chd_mean_abs_ig"]):
            color_chd = cmap(norm(row["chd_mean_abs_ig"]))
        else:
            color_chd = "#FFFFFF"#"#F7F7F7"

        if row["istroke_q75_region"] and pd.notna(row["istroke_mean_abs_ig"]):
            color_ist = cmap(norm(row["istroke_mean_abs_ig"]))
        else:
            color_ist = "#FFFFFF"
        ax.add_patch(
            Rectangle(
                (x, chd_track_y),
                w,
                TRACK_HEIGHT,
                facecolor=color_chd,
                edgecolor="white",
                linewidth=0.12,
                zorder=2
            )
        )

        ax.add_patch(
            Rectangle(
                (x, istroke_track_y),
                w,
                TRACK_HEIGHT,
                facecolor=color_ist,
                edgecolor="white",
                linewidth=0.12,
                zorder=2
            )
        )

    
    ax.add_patch(
        Rectangle(
            (0, chd_track_y),
            chrom_len_dict[chrom],
            TRACK_HEIGHT,
            facecolor="none",
            edgecolor=EDGE_COLOR,
            linewidth=0.6,
            zorder=3
        )
    )
    ax.add_patch(
        Rectangle(
            (0, istroke_track_y),
            chrom_len_dict[chrom],
            TRACK_HEIGHT,
            facecolor="none",
            edgecolor=EDGE_COLOR,
            linewidth=0.6,
            zorder=3
        )
    )

    
    ax.text(-6, y0, chrom, ha="right", va="center", fontsize=9)
    ax.text(-6, chd_track_y + TRACK_HEIGHT / 2, "CHD", ha="right", va="center", fontsize=8)
    ax.text(-6, istroke_track_y + TRACK_HEIGHT / 2, "istroke", ha="right", va="center", fontsize=8)

    
    spans_this_chrom = [s for s in label_region_spans if s["chrom"] == chrom]

    for span in spans_this_chrom:
        x = span["start_mb"]
        w = span["end_mb"] - span["start_mb"]

        box_y = y0 - CHROM_HEIGHT / 2 - 0.01
        box_h = (istroke_track_y + TRACK_HEIGHT) - box_y + 0.01

        ax.add_patch(
            Rectangle(
                (x, box_y),
                w,
                box_h,
                facecolor="none",
                edgecolor=LABEL_EDGE_COLOR,
                linewidth=LABEL_LINEWIDTH,
                zorder=6
            )
        )

        k = label_offset_counter[chrom]
        label_offset_counter[chrom] += 1

        text_x = x + w / 2
        text_y = y0 - CHROM_HEIGHT / 2 - 0.08 - 0.11 * (k % 3)

        ax.plot(
            [x + w / 2, x + w / 2],
            [box_y, text_y + 0.02],
            color=LABEL_EDGE_COLOR,
            linewidth=0.9,
            zorder=6
        )

        ax.text(
            text_x,
            text_y,
            span["label"],
            color=LABEL_TEXT_COLOR,
            fontsize=LABEL_FONTSIZE,
            ha="center",
            va="bottom",
            zorder=7
        )

# ======================================================
# Legends
# ======================================================

legend_handles = [
    Patch(facecolor=COLOR_CHD, edgecolor="none", label="Selected CHD study region"),
    Patch(facecolor=COLOR_ISTROKE, edgecolor="none", label="Selected istroke study region"),
    Patch(facecolor=COLOR_BOTH, edgecolor="none", label="Shared selected study region"),
    Patch(facecolor="#F7F7F7", edgecolor=EDGE_COLOR, label="Non-q75 heatmap region"),
    Patch(facecolor="white", edgecolor=LABEL_EDGE_COLOR, label="Highlighted labeled region"),
]

ax.legend(
    handles=legend_handles,
    frameon=False,
    fontsize=8,
    title="Regions",
    title_fontsize=9,
    loc="upper left",
    bbox_to_anchor=(1.02, 0.98)
)

# ======================================================
# Style
# ======================================================
'''
ax.set_title(
    "Selected chromosomes with manually selected GWAS study regions and dual SNP IG heatmaps\n"
    "Only q75 IG regions are color-filled in the heatmap; key loci are outlined in red",
    fontsize=13,
    pad=18
)
'''
ax.set_xlim(-12, max_chr_len + 8)

y_min = -0.80
y_max = (
    max(y_positions.values())
    + CHROM_HEIGHT
    + CHROM_TO_TRACK_GAP
    + 2 * TRACK_HEIGHT
    + TRACK_GAP
    + 0.75
)
ax.set_ylim(y_max, y_min)

ax.set_xlabel("Genomic position (Mb)")
ax.set_yticks([])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True, bbox_inches="tight")
plt.close()

print("Saved final figure:", OUTFIG)

# ======================================================
# Separate colorbar figure
# ======================================================

fig_cb, ax_cb = plt.subplots(figsize=(1.2, 3.2))

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = plt.colorbar(
    sm,
    cax=ax_cb,
    orientation="vertical"
)


cb_ticks = np.linspace(vmin, vmax, 3)
cbar.set_ticks(cb_ticks)
cbar.set_ticklabels([f"{x:.4f}" for x in cb_ticks])

cbar.set_label("Mean absolute IG", fontsize=10)
cbar.ax.tick_params(labelsize=8)

plt.savefig(
    OUT_COLORBAR,
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)

plt.close(fig_cb)

print("Saved separate colorbar:", OUT_COLORBAR)

# ======================================================
# Optional print
# ======================================================

print("\nQ75 selected regions:")
print(
    q75_table[
        [
            "chrom",
            "band_name",
            "chd_mean_abs_ig",
            "istroke_mean_abs_ig",
            "chd_q75_region",
            "istroke_q75_region",
            "region_type",
        ]
    ].to_string(index=False)
)

print("\nLabeled regions in final plot:")
for s in label_region_spans:
    print(
        f"{s['chrom']} {s['start_band']}-{s['end_band']} "
        f"({s['start_mb']:.2f}-{s['end_mb']:.2f} Mb): {s['label']} "
        f"[included bands: {s['included_bands']}]"
    )

In [ ]:
# ======================================================
# Print selected CHD / istroke GWAS studies in q75 selected bands
# ======================================================

OUT_SELECTED_STUDIES = os.path.join(
    OUTDIR,
    "CHD_istroke_q75_selected_GWAS_studies.csv"
)


selected_chd_band_ids = set(
    q75_table.loc[q75_table["chd_q75_region"], "band_id"]
)

selected_istroke_band_ids = set(
    q75_table.loc[q75_table["istroke_q75_region"], "band_id"]
)


band_info_cols = [
    "band_id",
    "chrom",
    "band_name",
    "chd_mean_abs_ig",
    "istroke_mean_abs_ig",
    "chd_q75_region",
    "istroke_q75_region",
    "region_type",
]

band_info = q75_table[band_info_cols].drop_duplicates()

selected_study_rows = []

# -----------------------------
# CHD selected studies
# -----------------------------
chd_studies = gwas_band[
    gwas_band["is_chd_related"] &
    gwas_band["band_id"].isin(selected_chd_band_ids)
].copy()

if not chd_studies.empty:
    chd_studies["endpoint"] = "CHD"
    selected_study_rows.append(chd_studies)

# -----------------------------
# istroke selected studies
# -----------------------------
istroke_studies = gwas_band[
    gwas_band["is_istroke_related"] &
    gwas_band["band_id"].isin(selected_istroke_band_ids)
].copy()

if not istroke_studies.empty:
    istroke_studies["endpoint"] = "istroke"
    selected_study_rows.append(istroke_studies)

# -----------------------------
# Combine and summarize
# -----------------------------
if len(selected_study_rows) == 0:

    print("\nNo CHD / istroke GWAS studies found in q75 selected bands.")

else:

    selected_studies = pd.concat(selected_study_rows, ignore_index=True)

    selected_studies = selected_studies.merge(
        band_info,
        on="band_id",
        how="left",
        suffixes=("", "_band")
    )

    
    keep_cols = [
        "endpoint",
        "chrom",
        "band_name",
        "band_id",
        "DISEASE/TRAIT",
        "STUDY",
        "FIRST AUTHOR",
        "DATE",
        "JOURNAL",
        "PUBMEDID",
        "SNPS",
        "MAPPED_GENE",
        "P-VALUE",
        "OR or BETA",
        "chd_mean_abs_ig",
        "istroke_mean_abs_ig",
        "region_type",
    ]

    keep_cols = [c for c in keep_cols if c in selected_studies.columns]

    selected_studies_out = selected_studies[keep_cols].copy()

    
    if "P-VALUE" in selected_studies_out.columns:
        selected_studies_out["P-VALUE"] = pd.to_numeric(
            selected_studies_out["P-VALUE"],
            errors="coerce"
        )

        selected_studies_out = selected_studies_out.sort_values(
            ["endpoint", "chrom", "band_name", "P-VALUE"],
            ascending=[True, True, True, True]
        )
    else:
        selected_studies_out = selected_studies_out.sort_values(
            ["endpoint", "chrom", "band_name"],
            ascending=[True, True, True]
        )

    selected_studies_out.to_csv(OUT_SELECTED_STUDIES, index=False)

    print("\nSaved selected CHD / istroke GWAS studies:")
    print(OUT_SELECTED_STUDIES)

    print("\nSelected GWAS studies in q75 bands:")
    print(
        selected_studies_out[
            [
                c for c in [
                    "endpoint",
                    "chrom",
                    "band_name",
                    "DISEASE/TRAIT",
                    "FIRST AUTHOR",
                    "DATE",
                    "JOURNAL",
                    "PUBMEDID",
                    "SNPS",
                    "MAPPED_GENE",
                    "P-VALUE",
                ]
                if c in selected_studies_out.columns
            ]
        ].to_string(index=False)
    )

    # -----------------------------
    
    # -----------------------------
    summary_group_cols = [
        c for c in [
            "endpoint",
            "chrom",
            "band_name",
            "DISEASE/TRAIT",
            "STUDY",
            "FIRST AUTHOR",
            "DATE",
            "JOURNAL",
            "PUBMEDID",
        ]
        if c in selected_studies_out.columns
    ]

    selected_study_summary = (
        selected_studies_out
        .groupby(summary_group_cols, dropna=False)
        .agg(
            n_gwas_hits=("SNPS", "count") if "SNPS" in selected_studies_out.columns else ("DISEASE/TRAIT", "count"),
            min_pvalue=("P-VALUE", "min") if "P-VALUE" in selected_studies_out.columns else ("DISEASE/TRAIT", "count")
        )
        .reset_index()
    )

    selected_study_summary = selected_study_summary.sort_values(
        ["endpoint", "chrom", "band_name", "min_pvalue"],
        ascending=[True, True, True, True]
    )

    OUT_SELECTED_STUDY_SUMMARY = os.path.join(
        OUTDIR,
        "CHD_istroke_q75_selected_GWAS_study_summary.csv"
    )

    selected_study_summary.to_csv(OUT_SELECTED_STUDY_SUMMARY, index=False)

    print("\nSaved selected study summary:")
    print(OUT_SELECTED_STUDY_SUMMARY)

    print("\nSelected GWAS study summary:")
    print(selected_study_summary.to_string(index=False))

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ======================================================
# Config
# ======================================================

TARGET_SNP = "rs3782805"
WINDOW_BP = 5000
ENDPOINT = "istroke"

IG_CSV = f"code_02_02_save/snp_20260202_1432/explanations/{ENDPOINT}_snp_TR_best_IG.csv"
PRED_CSV = f"code_02_02_save/snp_20260202_1432/preds/{ENDPOINT}_snp_TR_test_best_pred.csv"

VARIANT_CSV = "data/private/CHD_SNP_variant_gene_with_symbol.csv"
GWAS_CATALOG = "data/private/gwas-catalog-download-associations-v1.0-full.tsv"

HIGH_RISK_Q = 0.85
MAX_TRAITS = 50

OUTDIR = "fig2_final/locus_plots"
os.makedirs(OUTDIR, exist_ok=True)

OUTFIG = f"{OUTDIR}/{TARGET_SNP}_{ENDPOINT}_locus_pretty.pdf"

sns.set_style("white")

# ======================================================
# Beautiful palette
# ======================================================

TRAIT_COLORS = [
    "#6EB8D2",  # deep red
    "#479D88",  # slate blue
    "#435385",  # teal green
    "#E6A083",  # muted purple
    "#8790B2"   # warm gold
]
# ======================================================
# Parse variant positions
# ======================================================

variant = pd.read_csv(VARIANT_CSV)

def parse_variant(v):

    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))

    if m:
        return m.group(1), int(m.group(2))

    return np.nan, np.nan

variant[["chr","pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant(x))
)

variant = variant.dropna(subset=["chr","pos"])

# ======================================================
# Target SNP
# ======================================================

target = variant[variant["SNP"] == TARGET_SNP]

target_chr = target["chr"].iloc[0]
target_pos = target["pos"].iloc[0]

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

# ======================================================
# Load IG
# ======================================================

ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)

pred["pid"] = pred["ID"]

ig = ig.merge(
    variant[["SNP","chr","pos"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)

ig["abs_ig"] = ig["ig_abs"].abs()

# high risk filtering
cut = pred["risk"].quantile(HIGH_RISK_Q)
high_pid = pred.loc[pred["risk"] >= cut, "pid"]

ig = ig[ig["pid"].isin(high_pid)]

# locus subset
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
]

# aggregate
snp_agg = (
    locus
    .groupby(["SNP","pos"],as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig":"mean_abs_ig"})
)

# relative position
snp_agg["rel_pos"] = snp_agg["pos"] - target_pos

# ======================================================
# Load GWAS catalog
# ======================================================

gwas = pd.read_csv(GWAS_CATALOG, sep="\t", low_memory=False)

gwas["CHR_ID"] = pd.to_numeric(gwas["CHR_ID"], errors="coerce")
gwas["CHR_POS"] = pd.to_numeric(gwas["CHR_POS"], errors="coerce")

chr_numeric = int(target_chr.replace("chr",""))

gwas_region = gwas[
    (gwas["CHR_ID"] == chr_numeric) &
    (gwas["CHR_POS"] >= left) &
    (gwas["CHR_POS"] <= right)
]

# choose top traits
trait_counts = gwas_region["DISEASE/TRAIT"].value_counts()

top_traits = trait_counts.head(MAX_TRAITS).index.tolist()

gwas_region = gwas_region[
    gwas_region["DISEASE/TRAIT"].isin(top_traits)
]

gwas_region["rel_pos"] = gwas_region["CHR_POS"] - target_pos


In [ ]:
gwas_region

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from scipy.stats import mannwhitneyu

# ======================================================
# PDF / SVG font settings
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")

# ======================================================
# Config
# ======================================================

SOMA_IG_CSV_CHD = (
    "code_02_02_save/soma_20260202_2006/"
    "explanations/CHD_soma_TR_best_IG.csv"
)

PRED_CSV_CHD = (
    "code_02_02_save/soma_20260202_2006/"
    "preds/CHD_soma_TR_test_best_pred.csv"
)

SOMA_IG_CSV_IST = (
    "code_02_02_save/soma_20260202_2006/"
    "explanations/istroke_soma_TR_best_IG.csv"
)

PRED_CSV_IST = (
    "code_02_02_save/soma_20260202_2006/"
    "preds/istroke_soma_TR_test_best_pred.csv"
)

OUTDIR = "fig2_final"
os.makedirs(OUTDIR, exist_ok=True)

HIGH_RISK_Q = 0.95
TOP_N = 15
MAX_X = 0.0025

OUT_CHD = os.path.join(OUTDIR, "CHD_top_protein_boxplot_with_sig.pdf")
OUT_IST = os.path.join(OUTDIR, "istroke_top_protein_boxplot_with_sig.pdf")

# ===== box colors: light =====
BOX_PALETTE = {
    "High risk": "#F3B0B2",
    "Others": "#AFC7E8"
}

# ===== point colors: dark =====
POINT_PALETTE = {
    "High risk": "#C44E52",
    "Others": "#4C72B0"
}


# ======================================================
# Helper
# ======================================================

def parse_protein(feature):
    return str(feature).replace("seq.", "")


def p_to_star(p):

    if pd.isna(p):
        return "n.s."
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "n.s."


def add_sig_for_each_protein(
    ax,
    plot_df,
    protein_order,
    x_col="abs_ig",
    y_col="protein",
    group_col="risk_group",
    group1="High risk",
    group2="Others",
    max_x=0.0025
):

    """
    For each protein, compare High risk vs Others using Mann–Whitney U test.
    Add significance bracket and stars to each horizontal boxplot row.
    """

    for i, protein in enumerate(protein_order):

        sub = plot_df[plot_df[y_col] == protein].copy()

        vals1 = sub.loc[sub[group_col] == group1, x_col].dropna().values
        vals2 = sub.loc[sub[group_col] == group2, x_col].dropna().values

        if len(vals1) < 2 or len(vals2) < 2:
            pval = np.nan
        else:
            _, pval = mannwhitneyu(
                vals1,
                vals2,
                alternative="two-sided"
            )

        star = p_to_star(pval)

        local_max = sub[x_col].max()

        if pd.isna(local_max):
            local_max = 0

        # significance bracket position
        x0 = min(local_max + max_x * 0.035, max_x * 0.84)
        x1 = min(local_max + max_x * 0.075, max_x * 0.89)
        x_text = min(local_max + max_x * 0.090, max_x * 0.925)

        # For horizontal boxplot with hue dodge,
        # the two boxes are approximately at y +/- 0.20
        y1 = i - 0.20
        y2 = i + 0.20
        y_mid = i

        ax.plot(
            [x0, x1, x1, x0],
            [y1, y1, y2, y2],
            color="black",
            linewidth=0.8,
            clip_on=False
        )

        ax.text(
            x_text,
            y_mid,
            star,
            va="center",
            ha="left",
            fontsize=10,
            color="black"
        )


# ======================================================
# Load data
# ======================================================

def load_data(ig_csv, pred_csv):

    ig = pd.read_csv(ig_csv)
    pred = pd.read_csv(pred_csv)

    pred["pid"] = pred["ID"]

    if "ig_abs" in ig.columns:
        ig["abs_ig"] = ig["ig_abs"].abs()
    else:
        ig["abs_ig"] = ig["ig"].abs()

    ig["protein"] = ig["feature"].apply(parse_protein)

    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)

    pred["risk_group"] = np.where(
        pred["risk"] >= risk_cut,
        "High risk",
        "Others"
    )

    ig = ig.merge(
        pred[["pid", "risk_group"]],
        on="pid",
        how="inner"
    )

    return ig


# ======================================================
# Top proteins
# ======================================================

def compute_top_proteins(ig):

    protein_sample = (
        ig
        .groupby(["pid", "protein", "risk_group"], as_index=False)["abs_ig"]
        .max()
    )

    protein_rank = (
        protein_sample[protein_sample["risk_group"] == "High risk"]
        .groupby("protein", as_index=False)["abs_ig"]
        .median()
        .sort_values("abs_ig", ascending=False)
    )

    top_proteins = protein_rank["protein"].head(TOP_N).tolist()

    plot_df = protein_sample[
        protein_sample["protein"].isin(top_proteins)
    ].copy()

    protein_order = (
        protein_rank
        .set_index("protein")
        .loc[top_proteins]
        .sort_values("abs_ig", ascending=False)
        .index
        .tolist()
    )

    return plot_df, protein_order


# ======================================================
# Draw boxplot
# ======================================================

def draw_boxplot(plot_df, protein_order, title, outfile):

    fig_h = 0.4 * len(protein_order) + 1

    fig, ax = plt.subplots(
        figsize=(4.8, 4.2)
    )

    sns.boxplot(
        data=plot_df,
        x="abs_ig",
        y="protein",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=protein_order,
        orient="h",
        showfliers=False,
        palette=BOX_PALETTE,
        linewidth=1,
        width=0.6,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="abs_ig",
        y="protein",
        hue="risk_group",
        hue_order=["High risk", "Others"],
        order=protein_order,
        orient="h",
        dodge=True,
        jitter=0.25,
        size=3,
        alpha=0.9,
        edgecolor="black",
        linewidth=0.3,
        palette=POINT_PALETTE,
        ax=ax
    )

    # remove duplicated legends
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    # x axis name
    ax.set_xlabel("IG")
    ax.set_ylabel("")

    # optional title
    # ax.set_title(title, fontsize=11)

    # add significance
    add_sig_for_each_protein(
        ax=ax,
        plot_df=plot_df,
        protein_order=protein_order,
        x_col="abs_ig",
        y_col="protein",
        group_col="risk_group",
        group1="High risk",
        group2="Others",
        max_x=MAX_X+0.001
    )
    if "Coronary" in title:
        ax.set_xticks([0, 0.002, 0.004])
    else:
        ax.set_xticks([0, 0.001, 0.002])

    # clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    plt.savefig(
        outfile,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", outfile)


# ======================================================
# Run CHD
# ======================================================

ig_chd = load_data(
    SOMA_IG_CSV_CHD,
    PRED_CSV_CHD
)

plot_chd, order_chd = compute_top_proteins(
    ig_chd
)

draw_boxplot(
    plot_chd,
    order_chd,
    "Coronary Heart Disease",
    OUT_CHD
)


# ======================================================
# Run iStroke
# ======================================================

ig_ist = load_data(
    SOMA_IG_CSV_IST,
    PRED_CSV_IST
)

plot_ist, order_ist = compute_top_proteins(
    ig_ist
)

draw_boxplot(
    plot_ist,
    order_ist,
    "Ischemic Stroke",
    OUT_IST
)

print("All done.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# =========================
# Hard-coded GO rows (FINAL)
# =========================
rows = [
    # ================= CHD =================
    dict(
        Term="cholesterol\nhomeostasis",
        Adjusted_P=0.0125782520805777,
        Genes="APOM;LPL;PCSK9",
        dataset="CHD"
    ),
    dict(
        Term="regulation of\ncomplement activation",
        Adjusted_P=0.0081382172360859,
        Genes="C5;C1S;C2",
        dataset="CHD"
    ),
    dict(
        Term="extracellular matrix\norganization",
        Adjusted_P=0.0334762073457627,
        Genes="VCAM1",
        dataset="CHD"
    ),

    # ================= iStroke =================
    dict(
        Term="axon guidance",
        Adjusted_P=0.0021764350929948,
        Genes="ROBO2;CXCL12;ALCAM;PDLIM7",
        dataset="iStroke"
    ),
    dict(
        Term="neuron migration",
        Adjusted_P=0.0288136903132404,
        Genes="NTRK2;DNER",
        dataset="iStroke"
    ),
    dict(
        Term="synapse assembly",
        Adjusted_P=0.0450663671610746,
        Genes="DNER;PLXNB2",
        dataset="iStroke"
    )
]

'''
    dict(
        Term="regulation of protein\nkinase B signaling",
        Adjusted_P=0.0021764350929948,
        Genes="FGF5;NTRK2;MET;EGFR;IGF1R",
        dataset="iStroke"
    ),
'''
'''
    dict(
        Term="cell-cell adhesion via\nplasma-membrane\nadhesion molecules",
        Adjusted_P=0.0272784352173614,
        Genes="ROBO2;ALCAM",
        dataset="iStroke"
    ),
'''
'''
    dict(
        Term="endothelial cell\nmorphogenesis",
        Adjusted_P=0.0081382172360859,
        Genes="COL18A1;MET",
        dataset="CHD"
    ),
'''
'''
    dict(
        Term="proteolysis",
        Adjusted_P=0.0081382172360859,
        Genes="CPM;F9;ADAMTS13;PLG",
        dataset="CHD"
    ),
'''
df = pd.DataFrame(rows)

# =========================
# Output + style
# =========================
OUTDIR = Path("fig2_final/GO_selected_hardcoded_SOMA")
OUTDIR.mkdir(parents=True, exist_ok=True)

COLORS = {"CHD": "#808ba8", "iStroke": "#e29379"}

BAR_HEIGHT = 0.85
Y_SPACING = 1.55
FIG_W = 6.4
ROW_H = 1.15

XMAX = 3.0
XTICK_STEP = 0.5
GENE_X0 = 0.03
LEFT_MARGIN = 0.62


def plot_one(disease: str):
    sub = df[df["dataset"] == disease].copy()
    if sub.empty:
        print(f"No rows for {disease}")
        return

    sub["x"] = -np.log10(sub["Adjusted_P"].astype(float))
    sub = sub.sort_values("x", ascending=False).reset_index(drop=True)

    y = np.arange(len(sub)) * Y_SPACING
    fig_h = max(2.8, ROW_H * len(sub))

    plt.figure(figsize=(FIG_W, fig_h))
    ax = plt.gca()

    # Bars
    ax.barh(
        y=y,
        width=sub["x"].values,
        left=0.0,
        color=COLORS[disease],
        height=BAR_HEIGHT,
        zorder=1
    )

    # GO term labels (left of spine)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["Term"].tolist(), fontsize=11)
    for lab in ax.get_yticklabels():
        lab.set_ha("right")
        lab.set_va("center")
        lab.set_multialignment("right")

    # Gene labels inside bars
    for yi, genes in zip(y, sub["Genes"].tolist()):
        ax.text(
            GENE_X0,
            yi,
            genes.replace(";", ", "),
            ha="left",
            va="center",
            fontsize=9,
            color="black",
            zorder=3,
            clip_on=True
        )

    # X axis
    ax.set_xlim(0, XMAX)
    xticks = np.arange(0, XMAX + 1e-9, XTICK_STEP)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{t:.1f}" for t in xticks], fontsize=11)
    ax.set_xlabel(r"$-\log_{10}(\mathrm{FDR})$", fontsize=12)

    title = "Coronary Heart Disease" if disease == "CHD" else "Ischemic Stroke"
    ax.set_title(f"{title} – GO terms", fontsize=14)

    # Spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.6)
    ax.spines["bottom"].set_linewidth(1.6)
    ax.tick_params(axis="both", width=1.4, length=6)

    ax.grid(False)
    ax.invert_yaxis()
    plt.subplots_adjust(left=LEFT_MARGIN, right=0.98, top=0.90, bottom=0.14)

    out = OUTDIR / f"{disease}_GO_selected_barplot.pdf"
    plt.savefig(out, dpi=300, bbox_inches="tight", transparent=True)
    plt.close()
    print(f"Saved: {out}")


# =========================
# Run
# =========================
plot_one("CHD")
plot_one("iStroke")
print("Done.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from adjustText import adjust_text

# ======================================
# PATHS
# ======================================

CHD_FILE = "shared_gene_tables/CHD_shared_SNP_SOMA_gene_rank.csv"
IST_FILE = "shared_gene_tables/iStroke_shared_SNP_SOMA_gene_rank.csv"

OUTFIG = "fig2_final/Fig_CHD_iStroke_SNP_vs_SOMA_final.pdf"
OUTSVG = "fig2_final/Fig_CHD_iStroke_SNP_vs_SOMA_final.svg"

os.makedirs(os.path.dirname(OUTFIG), exist_ok=True)

# ======================================
# MATPLOTLIB SETTINGS
# ======================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

# ======================================
# PARAMETERS
# ======================================

TOP15 = 0.75
CUTOFF = 0.4

LOW_MAX = 0.75
LOW_SCALE = 0.25
HIGH_SCALE = 0.75

AX_MAX = 1.08

COLOR_BOTH = "#B9481F"
COLOR_SNP = "#479D88"
COLOR_SOMA = "#435385"
COLOR_BG = "#CFCFCF"

SIZE_BG = 18
SIZE_BOTH = 48
SIZE_SNP = 38
SIZE_SOMA = 38

SIZE_LABEL_BOTH = 115
SIZE_LABEL_SNP = 90
SIZE_LABEL_SOMA = 90
SIZE_LABEL_BG = 55

ALPHA_BG = 0.14
ALPHA_BOTH = 0.42
ALPHA_SNP = 0.38
ALPHA_SOMA = 0.38
ALPHA_LABEL = 0.98

# ======================================
# FINAL LABEL SET
# ======================================

CHD_LABEL_GENES = [
    "LPL", "TNFSF11", "TNFRSF10B", "FLT1", "COL15A1", "IGF2R",
    "CTSS", "CSF1R", "IL1RN", "VCAM1", "PCSK9",
    "CLEC1B", "VEGFA", "C3", "NPPB", "FLT3"
]

IST_LABEL_GENES = [
    "SPON1", "ACAN", "IGF1R", "CLEC1B", "NCK2",
    "CTSS", "LRIG3", "CD36", "CSF1R", "PCSK9", "NPPB",
    "TNFRSF19", "APOD", "ROBO2", "COL5A1", "LPL", "MRC1", "FLT1"
]

# ======================================
# AXIS TRANSFORM
# ======================================

def transform_axis(v):
    v = np.array(v, dtype=float)
    out = np.zeros_like(v)

    low = v <= LOW_MAX
    high = v > LOW_MAX

    out[low] = (v[low] - CUTOFF) / (LOW_MAX - CUTOFF) * LOW_SCALE
    out[high] = LOW_SCALE + (v[high] - LOW_MAX) / (1 - LOW_MAX) * HIGH_SCALE

    return out


def transform_tick(v):
    return transform_axis([v])[0]


# ======================================
# LOAD DATA
# ======================================

chd = pd.read_csv(CHD_FILE)
ist = pd.read_csv(IST_FILE)

# ======================================
# PREPARE DATA
# ======================================

def prepare(df):
    df = df.copy()

    df["snp_pct"] = df["snp_reverse_rank"] / df["snp_reverse_rank"].max()
    df["soma_pct"] = df["soma_reverse_rank"] / df["soma_reverse_rank"].max()

    df["snp_top"] = df["snp_pct"] >= TOP15
    df["soma_top"] = df["soma_pct"] >= TOP15

    df = df[
        (df["snp_pct"] >= CUTOFF) &
        (df["soma_pct"] >= CUTOFF)
    ].copy()

    df["x"] = transform_axis(df["soma_pct"])
    df["y"] = transform_axis(df["snp_pct"])

    return df


chd = prepare(chd)
ist = prepare(ist)

# ======================================
# LABEL FUNCTION
# ======================================

def annotate_genes(ax, df, label_genes):
    """Analysis helper. Inputs and outputs follow the function implementation."""

    label_df = df[df["gene"].isin(label_genes)].copy()

    if label_df.empty:
        return

    texts = []
    point_xy = []

    
    offsets = [
        (0.050,  0.040),
        (-0.050, 0.040),
        (0.050, -0.040),
        (-0.050,-0.040),
        (0.070,  0.000),
        (-0.070, 0.000),
        (0.000,  0.060),
        (0.000, -0.060),
        (0.070,  0.030),
        (-0.070, 0.030),
        (0.070, -0.030),
        (-0.070,-0.030),
    ]

    for i, (_, r) in enumerate(label_df.iterrows()):
        dx, dy = offsets[i % len(offsets)]

        px = float(r["x"])
        py = float(r["y"])

        t = ax.text(
            px + dx,
            py + dy,
            str(r["gene"]),
            fontsize=10.5,
            ha="center",
            va="center",
            color="black",
            zorder=40,
            clip_on=False
        )

        texts.append(t)
        point_xy.append((px, py))

    
    adjust_text(
        texts,
        ax=ax,
        expand=(1.15, 1.20),
        force_text=(0.45, 0.50),
        force_static=(0.20, 0.20),
        force_pull=(0.01, 0.01),
        only_move={
            "text": "xy",
            "static": "xy",
            "explode": "xy",
            "pull": "xy"
        },
        ensure_inside_axes=False
    )

    # ==================================================
    
    # ==================================================
    # ==================================================
    
    
    # ==================================================
    
    
    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    
    for t, (px, py) in zip(texts, point_xy):
        tx, ty = t.get_position()
    
        
        p_disp = ax.transData.transform((px, py))
        c_disp = ax.transData.transform((tx, ty))
    
        
        bbox = t.get_window_extent(renderer=renderer).expanded(1.12, 1.35)
    
        x0, y0, x1, y1 = bbox.x0, bbox.y0, bbox.x1, bbox.y1
    
        dx = c_disp[0] - p_disp[0]
        dy = c_disp[1] - p_disp[1]
    
        candidates = []
    
        
        if abs(dx) > 1e-9:
            for xb in [x0, x1]:
                lam = (xb - p_disp[0]) / dx
                if 0 < lam < 1:
                    y_int = p_disp[1] + lam * dy
                    if y0 <= y_int <= y1:
                        candidates.append(lam)
    
        
        if abs(dy) > 1e-9:
            for yb in [y0, y1]:
                lam = (yb - p_disp[1]) / dy
                if 0 < lam < 1:
                    x_int = p_disp[0] + lam * dx
                    if x0 <= x_int <= x1:
                        candidates.append(lam)
    
        
        if len(candidates) > 0:
            lam = min(candidates)
            end_disp = p_disp + lam * np.array([dx, dy])
            ex, ey = ax.transData.inverted().transform(end_disp)
        else:
            
            ex = px + 0.82 * (tx - px)
            ey = py + 0.82 * (ty - py)
    
        ax.plot(
            [px, ex],
            [py, ey],
            color="black",
            lw=1.05,
            alpha=0.95,
            solid_capstyle="round",
            zorder=25,
            clip_on=False
        )










# ======================================
# PANEL FUNCTION
# ======================================

def plot_panel(ax, df, title, label_genes):
    df = df.copy()
    df["is_label"] = df["gene"].isin(label_genes)

    bg = df[~(df["snp_top"] | df["soma_top"])]

    both = df[(df["snp_top"]) & (df["soma_top"])]
    snp_only = df[(df["snp_top"]) & (~df["soma_top"])]
    soma_only = df[(~df["snp_top"]) & (df["soma_top"])]

    both_non = both[~both["is_label"]]
    both_lab = both[both["is_label"]]

    snp_non = snp_only[~snp_only["is_label"]]
    snp_lab = snp_only[snp_only["is_label"]]

    soma_non = soma_only[~soma_only["is_label"]]
    soma_lab = soma_only[soma_only["is_label"]]

    bg_non = bg[~bg["is_label"]]
    bg_lab = bg[bg["is_label"]]

    # background points
    ax.scatter(
        bg_non["x"], bg_non["y"],
        s=SIZE_BG,
        color=COLOR_BG,
        alpha=ALPHA_BG,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=1
    )

    # non-label colored points: more transparent
    ax.scatter(
        both_non["x"], both_non["y"],
        s=SIZE_BOTH,
        color=COLOR_BOTH,
        alpha=ALPHA_BOTH,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=4
    )

    ax.scatter(
        snp_non["x"], snp_non["y"],
        s=SIZE_SNP,
        color=COLOR_SNP,
        alpha=ALPHA_SNP,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=4
    )

    ax.scatter(
        soma_non["x"], soma_non["y"],
        s=SIZE_SOMA,
        color=COLOR_SOMA,
        alpha=ALPHA_SOMA,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=4
    )

    # label gene points: bigger and nearly opaque
    ax.scatter(
        both_lab["x"], both_lab["y"],
        s=SIZE_LABEL_BOTH,
        color=COLOR_BOTH,
        alpha=ALPHA_LABEL,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=12
    )

    ax.scatter(
        snp_lab["x"], snp_lab["y"],
        s=SIZE_LABEL_SNP,
        color=COLOR_SNP,
        alpha=ALPHA_LABEL,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=12
    )

    ax.scatter(
        soma_lab["x"], soma_lab["y"],
        s=SIZE_LABEL_SOMA,
        color=COLOR_SOMA,
        alpha=ALPHA_LABEL,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=12
    )

    
    ax.scatter(
        bg_lab["x"], bg_lab["y"],
        s=SIZE_LABEL_BG,
        color=COLOR_BG,
        alpha=0.85,
        edgecolor="none",
        linewidth=0,
        clip_on=False,
        zorder=10
    )

    # labels + forced manual guide lines
    annotate_genes(ax, df, label_genes)

    # threshold lines
    ax.axvline(
        transform_tick(0.75),
        color="lightgray",
        lw=0.75,
        zorder=0
    )
    ax.axhline(
        transform_tick(0.75),
        color="lightgray",
        lw=0.75,
        zorder=0
    )

    # diagonal
    ax.plot(
        [0, 1],
        [0, 1],
        ls="--",
        color="lightgray",
        lw=0.75,
        zorder=0
    )

    ticks = [0.4, 0.6, 0.8, 0.9, 1.0]
    tick_pos = [transform_tick(t) for t in ticks]

    ax.set_xticks(tick_pos)
    ax.set_xticklabels(ticks)

    ax.set_yticks(tick_pos)
    ax.set_yticklabels(ticks)

    ax.set_xlim(-0.04, AX_MAX + 0.05)
    ax.set_ylim(-0.04, AX_MAX + 0.05)

    ax.set_title(title, fontsize=13)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ======================================
# FIGURE
# ======================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.6, 3.8),
    sharex=True,
    sharey=True
)

plot_panel(
    axes[0],
    chd,
    "Coronary Heart Disease",
    CHD_LABEL_GENES
)

plot_panel(
    axes[1],
    ist,
    "Ischemic Stroke",
    IST_LABEL_GENES
)

axes[0].set_xlabel("SOMA importance percentile")
axes[1].set_xlabel("SOMA importance percentile")

axes[0].set_ylabel("SNP importance percentile")
axes[1].set_ylabel("SNP importance percentile")

plt.tight_layout()

plt.savefig(
    OUTFIG,
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)

plt.savefig(
    OUTSVG,
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)

plt.show()

print("Saved:", OUTFIG)
print("Saved:", OUTSVG)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import gseapy as gp
import os

# ======================================================
# PATH
# ======================================================

CHD_FILE = "shared_gene_tables/CHD_shared_SNP_SOMA_gene_rank.csv"
IST_FILE = "shared_gene_tables/iStroke_shared_SNP_SOMA_gene_rank.csv"

OUTDIR = "fig2_final"
os.makedirs(OUTDIR, exist_ok=True)

VENN_FIG = f"{OUTDIR}/Fig_CHD_iStroke_venn.pdf"

# ======================================================
# PARAM
# ======================================================

TOP15 = 0.8
CUTOFF = 0.5
COLOR_MAP = {
    "CHD_only": "#838EAC",   
    "Stroke_only": "#E29379",
    "Overlap": "#6cb0c9"     
}
COLOR_CHD = "#838EAC"
COLOR_IST = "#E29379"
COLOR_OVERLAP = "#6cb0c9"

# ======================================================
# LOAD
# ======================================================

chd = pd.read_csv(CHD_FILE)
ist = pd.read_csv(IST_FILE)

# ======================================================
# PREP
# ======================================================

def prepare(df):

    df = df.copy()

    df["snp_pct"] = df["snp_reverse_rank"] / df["snp_reverse_rank"].max()
    df["soma_pct"] = df["soma_reverse_rank"] / df["soma_reverse_rank"].max()

    df["snp_top"] = df["snp_pct"] >= TOP15
    df["soma_top"] = df["soma_pct"] >= TOP15

    df = df[
        (df["snp_pct"] >= CUTOFF) &
        (df["soma_pct"] >= CUTOFF)
    ].copy()

    return df

chd = prepare(chd)
ist = prepare(ist)

# ======================================================
# SELECT GENES
# ======================================================

def get_selected(df):
    return set(df[(df["snp_top"]) | (df["soma_top"])]["gene"].dropna())

chd_genes = get_selected(chd)
ist_genes = get_selected(ist)

overlap = chd_genes & ist_genes
chd_only = chd_genes - ist_genes
ist_only = ist_genes - chd_genes

print("CHD:", len(chd_genes))
print("IST:", len(ist_genes))
print("Overlap:", len(overlap))
print("Overlap:", overlap)

# ======================================================
# VENN
# ======================================================

plt.figure(figsize=(3.5,3.5))

venn = venn2(
    [chd_genes, ist_genes],
    set_labels=("CHD","iStroke")
)

venn.get_patch_by_id('10').set_color(COLOR_CHD)
venn.get_patch_by_id('01').set_color(COLOR_IST)
venn.get_patch_by_id('11').set_color(COLOR_OVERLAP)

for k in ['10','01','11']:
    if venn.get_patch_by_id(k):
        venn.get_patch_by_id(k).set_alpha(0.6)

plt.title("Gene overlap")

plt.tight_layout()
plt.savefig(VENN_FIG, dpi=300, transparent=True)
plt.close()

print("Saved:", VENN_FIG)

# ======================================================
# GO ENRICHMENT FUNCTION
# ======================================================

def run_go(gene_set, name):

    if len(gene_set) < 5:
        print(f"{name}: too few genes, skip")
        return

    enr = gp.enrichr(
        gene_list=list(gene_set),
        gene_sets="GO_Biological_Process_2023",
        organism="Human",
        outdir=None,
        cutoff=0.05
    )

    res = enr.results.copy()

    
    res["minus_log10_p"] = -np.log10(res["Adjusted P-value"] + 1e-12)

    
    res = res[~res["Term"].str.contains("regulation of biological process", na=False)]

    out_csv = f"{OUTDIR}/GO_{name}.csv"
    res.to_csv(out_csv, index=False)

    print(f"Saved GO: {out_csv}")


# ======================================================
# RUN GO
# ======================================================

run_go(chd_only, "CHD_only")
run_go(ist_only, "iStroke_only")
run_go(overlap, "Overlap")

print("All done.")

In [ ]:
### !/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ======================================================

# ======================================================

data = [
    ["Cellular Response To\nLow-Density Lipoprotein Particle Stimulus", 1.2766, "Stroke_only"],
    ["Positive Regulation Of Macrophage Derived \n Foam Cell Differentiation", 1.4726, "Stroke_only"],
    ["Artery Development", 1.3589, "Overlap"],
    ["Axon Guidance", 1.7603, "Overlap"],
    ["Regulation Of Cell Migration", 1.7603, "Overlap"],
    ["Positive Regulation Of Lipid Kinase Activity", 1.1958, "CHD_only"],
    ["Blood Vessel Morphogenesis", 1.6830, "CHD_only"],
    ["Extracellular Matrix Organization", 1.3831, "CHD_only"],
]

df = pd.DataFrame(data, columns=["Term", "score", "Subset"])

# ======================================================

# ======================================================

COLOR_MAP = {
    "CHD_only": "#838EAC",   
    "Stroke_only": "#E29379",
    "Overlap": "#6cb0c9"     
}

df["color"] = df["Subset"].map(COLOR_MAP)

# ======================================================

# ======================================================

#df = df.sort_values("score", ascending=True)

# ======================================================
# Plot
# ======================================================

plt.figure(figsize=(6.2, 3.2))

y_pos = np.arange(len(df))

bars = plt.barh(
    y_pos,
    df["score"],
    color=df["color"],
    alpha=0.9
)


plt.yticks(y_pos, df["Term"], fontsize=8)


plt.xlabel("-log10(adj P)", fontsize=11)


ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
"""Analysis helper. Inputs and outputs follow the function implementation."""
# ======================================================

# ======================================================

plt.xlim(0, df["score"].max() + 0.5)

plt.tight_layout()

plt.savefig(
    "fig2_final/GO_selected_barplot.pdf",
    dpi=300,
    transparent=True
)

plt.show()